# 零售場景的生成式 AI：從 Prompt 到可上線的評測流程

**Google Cloud & Generative AI Applications**

---

這份 notebook 要證明一件事：

> **生成式 AI 的難處不在叫模型，而在你怎麼知道它有沒有變好。**

我們會用一個真實的零售任務（商品文案生成）走完整條路：

| 節 | 內容 | 重點 |
|---|---|---|
| §1 | 商品資料與法規禁詞 | 規則要跟著商品屬性走 |
| §2 | Prompt v0 → v3，每版只加一件事 | Prompt 是規格書，不是咒語 |
| §3 | 三層評測，產出版本 × 指標對比表 | 怎麼證明它變好了 |
| §4 | 場景 B：評論洞察 | 同一套流程，反方向用 |
| §5 | 成本：實際花了多少，外推到 10 萬 SKU | 教公式，不背數字 |

**§3 的對比表是整份 notebook 的重點。** 其他都是為了讓那張表可信。

---

### 執行前須知

- 需要一個啟用了 Vertex AI API 與計費的 GCP 專案。
- 全部跑完的成本很低（見 §5 實際印出的數字），但**不是零**。
- 若沒有 GCP 專案，可在 §0 的 CONFIG 把 `USE_VERTEX` 改成 `False`
  並填入 AI Studio API key。

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **Demo 全長 12 分鐘（30:00–42:00）。開場前先 Run all 一次，讓輸出都在畫面上。**
>
> 先確認 CONFIG 的 `OFFLINE_MODE = True` —— 這是最容易忘的一項。
>
> 開場第一句（不要道歉，一句帶過）：
> 「我先說明，這些輸出是我昨天跑好存下來的，現在是重播 —— 因為會場網路我不敢賭。
> 程式碼跟評測都是當場執行的，只有呼叫模型那一步是查表。」

</div>

## §0 環境設定

In [ ]:
# Colab 已預裝 pandas / jinja2 / matplotlib，通常只需要 google-genai。
# 在乾淨環境（本機、其他 notebook 服務）請解除註解安裝完整清單：
#   §3 的對比表用 df.style.background_gradient() 上色，
#   .style 需要 jinja2、background_gradient 需要 matplotlib，缺一就會 AttributeError。
# %pip install -q google-genai pandas jinja2 matplotlib
# %pip install -q "google-cloud-bigquery[pandas]"   # 只有要跑 §4 的 BigQuery 才需要

### CONFIG — 只改這一格

價格常數**使用前必須到官方頁面現查更新**：
<https://cloud.google.com/vertex-ai/generative-ai/pricing>

模型與價格變動頻繁，任何寫死的數字都會很快過期。
要看實際花費，請以 §5 執行後印出來的金額為準。

In [ ]:
"""集中設定：模型、價格、通路限制。

所有環境相關設定集中在這裡。換專案、換模型、換價格都只改這個檔案。
價格必須到官方頁面現查後更新，不要沿用寫死的數字。
"""

# --- GCP ---
PROJECT_ID = "cacafly-poc"

# ⚠️ 必須是 "global"。實測 asia-east1 上沒有任何 Gemini publisher model，
#    所有呼叫都會 404。這是很容易踩的坑：compute 資源設在 asia-east1，
#    不代表 Gemini 模型在那裡可用。
LOCATION = "global"

# 使用 Vertex AI（走 GCP 專案計費、資料落地可控）
# 若現場 GCP 權限失效，把 USE_VERTEX 改成 False 並填入 AI Studio key 作為備援
USE_VERTEX = True
FALLBACK_API_KEY = ""  # 僅備援用，勿 commit 真實金鑰

# --- 離線重播（會場網路不穩時的主要策略）---
# 在網路不穩的場合（會議室、教室、展場）即時打 API 是不能接受的風險。
#
#   RECORD_FIXTURES = True   有網路時先跑一次，把真實輸出錄下來
#   OFFLINE_MODE    = True   之後零網路重播錄好的輸出
#
# 重播時所有 cell 一樣會執行、表格一樣是當場算出來的，只有 API 呼叫被換成
# 查表。畫面上看不出差別，但完全不依賴網路。
#
# ⚠ 兩個不能同時為 True。
OFFLINE_MODE = False
RECORD_FIXTURES = False
FIXTURES_FILE = "demo_outputs.json"

# --- 模型 ---
# 以下型號皆已在 cacafly-poc / global 實測可用（2026-08-13）。
# 換專案或換 region 前請重跑 poc/check_env.py 確認。
GEN_MODEL = "gemini-flash-latest"   # 生成用：吞吐量大、要便宜
JUDGE_MODEL = "gemini-2.5-pro"      # 評審用：刻意與生成模型不同，降低 self-preference bias
CHEAP_MODEL = "gemini-2.5-flash-lite"  # 成本段示範「降級」用

GEN_TEMPERATURE = 0.4  # 用低溫，降低失敗機率
JUDGE_TEMPERATURE = 0.0

# --- 價格（USD / 每百萬 token）---
# ⚠️ 這些是佔位值。使用前請到官方頁面現查更新：
#    https://cloud.google.com/vertex-ai/generative-ai/pricing
# 不要引用這裡的數字，要看 §5 實際跑出來的花費。
PRICING = {
    "gemini-flash-latest": {"input": 0.30, "output": 2.50},
    "gemini-2.5-pro": {"input": 1.25, "output": 10.00},
    "gemini-2.5-flash-lite": {"input": 0.10, "output": 0.40},
}
PRICE_LAST_CHECKED = "尚未查證 — 上場前必須更新"

# --- BigQuery（場景 B）---
# 預設關閉：重播模式不涵蓋 BigQuery 呼叫。
# 錄製時可開啟一次，把「AI 當 ETL」這件事真的做完。
USE_BIGQUERY = False
BQ_DATASET = "retail_genai_demo"
BQ_TABLE = "review_insights"
BQ_LOCATION = "asia-east1"  # BigQuery 的 region 與 Gemini 無關，這裡可以用亞洲

USD_TO_TWD = 32.0

# --- 通路限制（來自 products.json 的 _meta.channel_limits）---
SHOPEE_TITLE_MAX = 60
SEO_DESC_MAX = 120
BULLET_MAX = 30
BULLET_COUNT = 4

### 呼叫層與離線重播

底下這段程式碼做兩件事。

**第一，切換 Vertex AI 與 AI Studio 只差 Client 的建構參數**，
其他程式碼一行都不用改。選哪一個是**部署決策**
（資料落地、計費歸屬、VPC-SC、合規），不是**程式碼決策**。
先用 AI Studio 做原型，要簽 DPA 時再切到 Vertex。

**第二，離線重播。** 在網路不穩的場合即時打 API 是不能接受的風險，
所以有兩個模式：

| 設定 | 何時用 | 行為 |
|---|---|---|
| `RECORD_FIXTURES=True` | 有網路時，先跑一次 | 正常呼叫，同時把輸出錄下來 |
| `OFFLINE_MODE=True` | 之後任何時候 | 零網路，重播錄好的輸出 |

重播時**所有 cell 一樣會執行、表格一樣是當場算出來的**，
只有 API 呼叫被換成查表。`ReplayClient` 找不到對應輸出時會明確報錯，
不會靜默拿到過期資料。

In [ ]:
"""呼叫 Gemini 產生文案，並記錄 token 用量。

用 google-genai 統一 SDK。同一份程式碼可切換 Vertex AI 與 Gemini API ——
只差 Client 的建構參數。重點是：
選 Vertex 還是 AI Studio 是**部署決策**，不是**程式碼決策**。
"""

from __future__ import annotations

import time
from dataclasses import dataclass, field



@dataclass
class Usage:
    """單次呼叫的 token 用量。所有成本計算都從這裡出發。"""

    model: str
    input_tokens: int
    output_tokens: int
    latency_s: float

    @property
    def cost_usd(self) -> float:
        price = PRICING.get(self.model)
        if price is None:
            return 0.0
        return (
            self.input_tokens * price["input"] + self.output_tokens * price["output"]
        ) / 1_000_000


@dataclass
class GenResult:
    text: str
    usage: Usage
    error: str | None = None


@dataclass
class UsageLedger:
    """累計所有呼叫的用量 —— demo §5 成本計算的資料來源。

    刻意把 judge 的呼叫也記進來。評測本身要花錢，這是最常被漏算的一筆。
    """

    calls: list[tuple[str, Usage]] = field(default_factory=list)

    def record(self, tag: str, usage: Usage) -> None:
        self.calls.append((tag, usage))

    def total_cost_usd(self, tag_prefix: str | None = None) -> float:
        return sum(
            u.cost_usd
            for t, u in self.calls
            if tag_prefix is None or t.startswith(tag_prefix)
        )

    def total_tokens(self, tag_prefix: str | None = None) -> tuple[int, int]:
        rows = [u for t, u in self.calls if tag_prefix is None or t.startswith(tag_prefix)]
        return sum(u.input_tokens for u in rows), sum(u.output_tokens for u in rows)

    def summary(self) -> str:
        gen_in, gen_out = self.total_tokens("gen")
        jdg_in, jdg_out = self.total_tokens("judge")
        gen_cost = self.total_cost_usd("gen")
        jdg_cost = self.total_cost_usd("judge")
        total = gen_cost + jdg_cost
        judge_share = (jdg_cost / total * 100) if total else 0.0
        return (
            f"呼叫次數：{len(self.calls)}\n"
            f"  生成  input {gen_in:>7,} / output {gen_out:>7,} tokens → US${gen_cost:.4f}\n"
            f"  評審  input {jdg_in:>7,} / output {jdg_out:>7,} tokens → US${jdg_cost:.4f}\n"
            f"  合計                                        → US${total:.4f}"
            f"（約 NT${total * USD_TO_TWD:.2f}）\n"
            f"\n  ⚠ 評審佔總成本 {judge_share:.0f}% —— 這筆最常被漏算。"
        )


# --------------------------------------------------------------------------
# 離線重播 —— 會場網路不穩時的主要策略
# --------------------------------------------------------------------------
def fixture_key(model: str, contents: str, structured: bool) -> str:
    """以 (模型, prompt, 是否結構化) 當索引鍵。

    prompt 完全相同才會命中，所以只要 prompt 有任何改動，重播就會失敗並
    明確報錯 —— 不會靜默拿到舊資料。這是刻意的：寧可在測試時炸掉，
    也不要產出跟程式碼對不上的結果。
    """
    import hashlib

    raw = f"{model}\x00{int(structured)}\x00{contents}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]


class _ReplayModels:
    def __init__(self, fixtures: dict):
        self._f = fixtures
        self.hits = 0

    def generate_content(self, *, model, contents, config=None):
        structured = bool(config is not None and getattr(config, "response_schema", None))
        key = fixture_key(model, contents if isinstance(contents, str) else str(contents), structured)
        row = self._f.get("calls", {}).get(key)
        if row is None:
            raise KeyError(
                f"離線重播找不到對應輸出（key={key}, model={model}）。\n"
                f"代表 prompt 或模型在錄製之後被改過。\n"
                f"請重新錄製：把 RECORD_FIXTURES 設為 True、在有網路的環境跑一次。"
            )
        self.hits += 1
        return _Recorded(row["text"], row["input_tokens"], row["output_tokens"])

    def count_tokens(self, *, model, contents):
        key = "tok:" + fixture_key(model, contents, False)
        row = self._f.get("token_counts", {}).get(key)
        if row is None:
            raise KeyError(f"離線重播找不到 count_tokens 結果（key={key}）。請重新錄製。")
        self.hits += 1
        return _RecordedTokens(row)


class _Recorded:
    def __init__(self, text: str, pin: int, pout: int):
        self.text = text
        self.usage_metadata = _RecordedUsage(pin, pout)


class _RecordedUsage:
    def __init__(self, pin: int, pout: int):
        self.prompt_token_count = pin
        self.candidates_token_count = pout


class _RecordedTokens:
    def __init__(self, n: int):
        self.total_tokens = n


class ReplayClient:
    """從錄好的 fixtures 回放，完全不碰網路。"""

    def __init__(self, fixtures: dict):
        self.models = _ReplayModels(fixtures)


class _RecordingModels:
    def __init__(self, real):
        self._real = real
        self.fixtures = {"calls": {}, "token_counts": {}}

    def generate_content(self, *, model, contents, config=None):
        resp = self._real.generate_content(model=model, contents=contents, config=config)
        structured = bool(config is not None and getattr(config, "response_schema", None))
        key = fixture_key(model, contents if isinstance(contents, str) else str(contents), structured)
        meta = resp.usage_metadata
        self.fixtures["calls"][key] = {
            "text": resp.text or "",
            "input_tokens": getattr(meta, "prompt_token_count", 0) or 0,
            "output_tokens": getattr(meta, "candidates_token_count", 0) or 0,
        }
        return resp

    def count_tokens(self, *, model, contents):
        resp = self._real.count_tokens(model=model, contents=contents)
        self.fixtures["token_counts"]["tok:" + fixture_key(model, contents, False)] = (
            resp.total_tokens
        )
        return resp


class RecordingClient:
    """包住真實 client，一邊正常呼叫一邊把結果錄下來。"""

    def __init__(self, real_client):
        self.models = _RecordingModels(real_client.models)

    def dump(self) -> dict:
        return self.models.fixtures


def make_client(fixtures: dict | None = None):
    """建立 client。

    三種模式：
      OFFLINE_MODE    → ReplayClient，零網路，重播錄好的輸出（無網路時）
      RECORD_FIXTURES → RecordingClient，正常呼叫並錄下來（有網路時）
      預設             → 真實 client
    """
    if OFFLINE_MODE and RECORD_FIXTURES:
        raise RuntimeError("OFFLINE_MODE 與 RECORD_FIXTURES 不能同時為 True。")

    if OFFLINE_MODE:
        if not fixtures or not fixtures.get("calls"):
            raise RuntimeError(
                "OFFLINE_MODE=True 但沒有可用的 fixtures。\n"
                "請先在有網路的環境設定 RECORD_FIXTURES=True 跑一次，產生 "
                f"{FIXTURES_FILE}，再重新產生 notebook。"
            )
        return ReplayClient(fixtures)

    from google import genai

    if USE_VERTEX:
        real = genai.Client(
            vertexai=True,
            project=PROJECT_ID,
            location=LOCATION,
        )
    else:
        if not FALLBACK_API_KEY:
            raise RuntimeError(
                "USE_VERTEX=False 但 FALLBACK_API_KEY 是空的。"
                "請在 CONFIG cell 填入 AI Studio key，或改回 USE_VERTEX=True。"
            )
        real = genai.Client(api_key=FALLBACK_API_KEY)

    return RecordingClient(real) if RECORD_FIXTURES else real


def generate(
    client,
    prompt: str,
    *,
    structured: bool = False,
    model: str | None = None,
    temperature: float | None = None,
    max_retries: int = 3,
) -> GenResult:
    """單次生成。

    structured=True 時啟用 API 原生的 responseSchema —— 這是 v3 與 v2 的唯一差別。
    重點：不要用 prompt 硬凹 JSON 格式，要用 API 參數約束，模型才真的被限制在 schema 內。
    """
    from google.genai import types


    model = model or GEN_MODEL
    temperature = GEN_TEMPERATURE if temperature is None else temperature

    cfg_kwargs = {"temperature": temperature}
    if structured:
        cfg_kwargs["response_mime_type"] = "application/json"
        cfg_kwargs["response_schema"] = COPY_SCHEMA

    last_err = None
    for attempt in range(max_retries):
        started = time.time()
        try:
            resp = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(**cfg_kwargs),
            )
            meta = resp.usage_metadata
            return GenResult(
                text=resp.text or "",
                usage=Usage(
                    model=model,
                    input_tokens=getattr(meta, "prompt_token_count", 0) or 0,
                    output_tokens=getattr(meta, "candidates_token_count", 0) or 0,
                    latency_s=round(time.time() - started, 2),
                ),
            )
        except Exception as e:  # noqa: BLE001 — demo 要能撐過暫時性錯誤
            last_err = e
            if attempt < max_retries - 1:
                time.sleep(2**attempt)  # 指數退避

    return GenResult(
        text="",
        usage=Usage(model=model, input_tokens=0, output_tokens=0, latency_s=0.0),
        error=str(last_err),
    )


def count_tokens(client, text: str, model: str | None = None) -> int:
    """實測中文的 token 數。

    不要背「中文一個字約幾個 token」這種經驗法則 —— 用這個函式直接量。
    量出來的數字比任何通用經驗法則可靠。
    """
    model = model or GEN_MODEL
    resp = client.models.count_tokens(model=model, contents=text)
    return resp.total_tokens

In [ ]:
# ⚠ 尚未錄製 fixtures，OFFLINE_MODE 目前不可用。
#   請設定 RECORD_FIXTURES=True 在有網路時跑一次，
#   下載產生的 demo_outputs.json 放到 poc/data/，再重新產生 notebook。
FIXTURES = {'calls': {}, 'token_counts': {}}

In [ ]:
import json, time, re
from dataclasses import dataclass, field, asdict
import pandas as pd

client = make_client(FIXTURES)

if OFFLINE_MODE:
    print("✓ 離線重播模式 — 不會連網")
elif RECORD_FIXTURES:
    print("✓ 錄製模式 — 正常呼叫並記錄輸出")
    print(f"  {'Vertex AI' if USE_VERTEX else 'Gemini API'}")
else:
    print(f"✓ {'Vertex AI — project=' + PROJECT_ID if USE_VERTEX else 'Gemini API (AI Studio key)'}")

print(f"  生成模型：{GEN_MODEL}")
print(f"  評審模型：{JUDGE_MODEL}  ← 刻意與生成模型不同，降低 self-preference bias")

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **30:00 — 捲動帶過，不要逐行讀。**
>
> 要說的一句：「注意這幾行 —— 切 Vertex 或 AI Studio 只差 client 的參數，
> 底下所有程式碼一行都不用改。這是部署決策，不是技術決策。」

</div>

## §1 商品資料

12 筆模擬台灣電商 PIM 匯出的商品資料。刻意涵蓋三種法規類別：

- `food` — 保健食品／食品，受《食品安全衛生管理法》第 28 條規範
- `cosmetic` — 美妝保養，受《化粧品衛生安全管理法》第 10 條規範
- `general` — 3C／家電，無特殊廣告限制

**這個欄位是後面所有事情的關鍵。** 「護眼」用在葉黃素上是違規，
用在螢幕護目鏡上不是。規則必須跟著商品屬性走。

In [ ]:
PRODUCTS_RAW = {
 "_meta": {
  "description": "示範用假商品資料，模擬台灣電商 PIM 匯出格式。所有品牌、SKU、價格均為虛構，請勿對應真實商品。",
  "note": "regulated_category 決定套用哪一組法規禁詞：food=食安法, cosmetic=化粧品衛管法, general=無特殊限制",
  "channel_limits": {
   "shopee_title_max": 60,
   "momo_title_max": 50,
   "seo_desc_max": 120,
   "bullet_max": 30
  }
 },
 "products": [
  {
   "sku": "HB-1001",
   "name": "金盞花萃取葉黃素膠囊 60粒",
   "brand": "晨光研選",
   "category": "保健食品 / 眼部保養",
   "regulated_category": "food",
   "price": 890,
   "specs": {
    "淨含量": "60粒 / 盒",
    "主成分": "游離型葉黃素 30mg、玉米黃素 6mg",
    "劑型": "植物膠囊",
    "產地": "台灣",
    "建議食用": "每日1粒，隨餐食用"
   },
   "must_include_keywords": [
    "葉黃素",
    "60粒",
    "30mg"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "25-45 歲長時間使用螢幕的上班族"
  },
  {
   "sku": "HB-1002",
   "name": "五益菌複方益生菌粉 30入",
   "brand": "晨光研選",
   "category": "保健食品 / 消化保健",
   "regulated_category": "food",
   "price": 1180,
   "specs": {
    "淨含量": "30包 / 盒，每包 2g",
    "主成分": "五株專利益生菌，每包 100 億 CFU",
    "劑型": "粉末隨身包",
    "產地": "台灣",
    "建議食用": "每日1包，空腹或睡前"
   },
   "must_include_keywords": [
    "益生菌",
    "30包",
    "100億"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "外食族、作息不規律的上班族"
  },
  {
   "sku": "HB-1003",
   "name": "小分子魚膠原蛋白粉 蔓越莓風味 30入",
   "brand": "晨光研選",
   "category": "保健食品 / 美容保養",
   "regulated_category": "food",
   "price": 1580,
   "specs": {
    "淨含量": "30包 / 盒，每包 3g",
    "主成分": "深海魚膠原蛋白胜肽 2500mg、維生素C 80mg",
    "劑型": "粉末隨身包",
    "產地": "台灣",
    "建議食用": "每日1包，睡前溫水沖泡"
   },
   "must_include_keywords": [
    "膠原蛋白",
    "2500mg",
    "蔓越莓"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "30-50 歲注重外在保養的女性"
  },
  {
   "sku": "CS-2001",
   "name": "積雪草舒緩精華液 30ml",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 精華液",
   "regulated_category": "cosmetic",
   "price": 780,
   "specs": {
    "容量": "30ml",
    "主成分": "積雪草萃取 5%、泛醇 2%、神經醯胺",
    "質地": "水感精華",
    "適用膚質": "所有膚質，敏弱肌適用",
    "產地": "韓國"
   },
   "must_include_keywords": [
    "積雪草",
    "30ml",
    "5%"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "CS-2002",
   "name": "胺基酸氨基酸溫和洗面乳 120ml",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 清潔",
   "regulated_category": "cosmetic",
   "price": 420,
   "specs": {
    "容量": "120ml",
    "主成分": "胺基酸界面活性劑、洋甘菊萃取",
    "質地": "綿密泡沫",
    "pH值": "弱酸性 pH5.5",
    "產地": "台灣"
   },
   "must_include_keywords": [
    "胺基酸",
    "120ml",
    "pH5.5"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "CS-2003",
   "name": "多重玻尿酸保濕面膜 5片入",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 面膜",
   "regulated_category": "cosmetic",
   "price": 350,
   "specs": {
    "容量": "5片 / 盒，每片 25ml 精華",
    "主成分": "五重玻尿酸、海藻醣",
    "膜布": "天絲蠶絲膜布",
    "適用膚質": "乾燥、缺水肌",
    "產地": "台灣"
   },
   "must_include_keywords": [
    "玻尿酸",
    "5片",
    "天絲"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "EL-3001",
   "name": "無線降噪藍牙耳機 Pro 2",
   "brand": "Volta",
   "category": "3C / 音訊",
   "regulated_category": "general",
   "price": 3290,
   "specs": {
    "降噪深度": "最高 42dB 主動降噪",
    "續航": "單次 8 小時，含充電盒 32 小時",
    "連線": "藍牙 5.4，支援雙裝置切換",
    "防水": "IPX4",
    "重量": "單耳 4.8g"
   },
   "must_include_keywords": [
    "降噪",
    "32小時",
    "藍牙5.4"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "通勤族、遠距工作者"
  },
  {
   "sku": "EL-3002",
   "name": "65W 氮化鎵三孔快充充電器",
   "brand": "Volta",
   "category": "3C / 充電",
   "regulated_category": "general",
   "price": 990,
   "specs": {
    "輸出": "65W 總輸出，USB-C×2 + USB-A×1",
    "技術": "第三代氮化鎵 GaN",
    "尺寸": "5.2 × 5.2 × 2.9 cm",
    "重量": "112g",
    "保護": "過壓、過流、過熱保護"
   },
   "must_include_keywords": [
    "65W",
    "氮化鎵",
    "三孔"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "出差族、多裝置使用者"
  },
  {
   "sku": "EL-3003",
   "name": "手持無線吸塵器 Lite",
   "brand": "Volta",
   "category": "家電 / 清潔",
   "regulated_category": "general",
   "price": 4680,
   "specs": {
    "吸力": "18000Pa",
    "續航": "標準模式 35 分鐘",
    "集塵盒": "0.5L 可水洗",
    "重量": "1.4kg",
    "配件": "縫隙吸頭、床墊吸頭、軟絨滾筒"
   },
   "must_include_keywords": [
    "18000Pa",
    "35分鐘",
    "1.4kg"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "小坪數家庭、養寵物族群"
  },
  {
   "sku": "FD-4001",
   "name": "冷萃掛耳咖啡 淺焙水果調 10入",
   "brand": "溯源焙所",
   "category": "food / 咖啡",
   "regulated_category": "food",
   "price": 460,
   "specs": {
    "淨含量": "10包 / 盒，每包 12g",
    "產區": "衣索比亞 耶加雪菲",
    "烘焙度": "淺焙",
    "風味": "柑橘、莓果、花香",
    "保存": "室溫陰涼處，開封後盡快飲用"
   },
   "must_include_keywords": [
    "耶加雪菲",
    "10包",
    "淺焙"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "25-45 歲精品咖啡愛好者"
  },
  {
   "sku": "FD-4002",
   "name": "低溫烘焙綜合堅果 無調味 350g",
   "brand": "溯源焙所",
   "category": "food / 零食",
   "regulated_category": "food",
   "price": 590,
   "specs": {
    "淨含量": "350g / 罐",
    "內容": "腰果、核桃、杏仁、夏威夷豆",
    "製程": "低溫烘焙，無油炸",
    "調味": "完全無調味、無添加糖鹽",
    "保存": "開封後冷藏，兩週內食用完畢"
   },
   "must_include_keywords": [
    "綜合堅果",
    "350g",
    "無調味"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "注重原型食物的家庭與健身族群"
  },
  {
   "sku": "FD-4003",
   "name": "台灣紅烏龍茶包 三角立體茶包 20入",
   "brand": "溯源焙所",
   "category": "food / 茶葉",
   "regulated_category": "food",
   "price": 380,
   "specs": {
    "淨含量": "20包 / 盒，每包 3g",
    "產區": "台東鹿野",
    "發酵度": "重發酵",
    "風味": "蜜香、熟果、尾韻回甘",
    "沖泡": "95°C 熱水，浸泡 90 秒"
   },
   "must_include_keywords": [
    "紅烏龍",
    "鹿野",
    "20包"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "30-55 歲台灣茶愛好者、送禮需求"
  }
 ]
}

PRODUCTS = PRODUCTS_RAW['products']
CHANNEL_LIMITS = PRODUCTS_RAW['_meta']['channel_limits']

import pandas as pd
pd.DataFrame([{
    'SKU': p['sku'], '商品': p['name'][:22],
    '法規類別': p['regulated_category'], '售價': p['price'],
} for p in PRODUCTS])

### 禁詞清單

依台灣廣告法規整理，分成四組。**罰則差異很大**：

| 類別 | 法源 | 罰則 |
|---|---|---|
| 醫療效能 | 食安法 §28 第2項 | NT$60萬 – 500萬 |
| 不實誇張 | 食安法 §28 第1項 | NT$4萬 – 400萬 |
| 化粧品虛偽誇大 | 化粧品衛管法 §10 | NT$4萬 – 20萬 |

> 這不是工程潔癖，是不做會收罰單。這也是為什麼規則層要放在第一層。

⚠️ 本清單為示範用的簡化版，不構成法律意見；實際商用前應由法務或法規顧問審閱。

In [ ]:
BANNED_DATA = {
 "_meta": {
  "description": "商品文案禁詞清單，依台灣廣告法規分類。用於評測規則層。",
  "disclaimer": "本清單為示範用的簡化版，依主管機關公開認定準則的分類邏輯整理，不構成法律意見。實際商用前應由法務或法規顧問審閱，並以主管機關最新公告為準。",
  "legal_basis": {
   "food": {
    "law": "食品安全衛生管理法 第28條",
    "sub_regulation": "食品及相關產品標示宣傳廣告涉及不實誇張易生誤解或醫療效能認定準則",
    "penalty_exaggeration": "違反第1項（不實、誇張、易生誤解）：新臺幣 4 萬元以上 400 萬元以下罰鍰",
    "penalty_medical": "違反第2項（醫療效能）：新臺幣 60 萬元以上 500 萬元以下罰鍰",
    "criteria_ref": "認定準則 第4條（不實誇張易生誤解）、第5條（醫療效能）"
   },
   "cosmetic": {
    "law": "化粧品衛生安全管理法 第10條",
    "sub_regulation": "化粧品標示宣傳廣告涉及虛偽誇大或醫療效能認定準則",
    "note": "化粧品不得為虛偽或誇大之標示、宣傳、廣告，亦不得有醫療效能之標示、宣傳或廣告。",
    "penalty_exaggeration": "違反第1項（虛偽、誇大）：新臺幣 4 萬元以上 20 萬元以下罰鍰",
    "penalty_escalation": "若宣稱內容已使產品構成藥品，另可能違反藥事法（未經核准擅自製造販賣藥品／誇大醫療效能廣告），罰則顯著加重。說明此類升級風險時應指出其法源不同，勿逕稱『化粧品罰 500 萬』。"
   }
  },
  "important_nuance": "領有衛福部『健康食品』許可證（小綠人標章）的產品，得於許可範圍內宣稱經核可的保健功效。本清單假設商品『未取得』該許可，因此一律禁用。這正是規則層需要按商品屬性套用不同清單的原因。"
 },
 "medical_efficacy": {
  "_note": "涉及疾病之預防、改善、減輕、診斷或治療（認定準則第5條）。這是罰則最重的一類。",
  "applies_to": [
   "food",
   "cosmetic"
  ],
  "severity": "critical",
  "terms": [
   "治療",
   "療效",
   "醫療",
   "根治",
   "治癒",
   "痊癒",
   "藥效",
   "特效",
   "預防疾病",
   "抗癌",
   "防癌",
   "抑制腫瘤",
   "降血壓",
   "降血糖",
   "降膽固醇",
   "降尿酸",
   "降血脂",
   "消炎",
   "抗發炎",
   "殺菌",
   "抗菌",
   "抗病毒",
   "改善過敏",
   "治療過敏",
   "根治過敏",
   "改善失眠",
   "治療失眠",
   "護肝",
   "解毒",
   "排毒",
   "增強免疫力",
   "提升免疫力",
   "調節免疫",
   "抗憂鬱",
   "改善憂鬱",
   "壯陽",
   "生髮",
   "治療掉髮",
   "改善關節炎",
   "治療骨質疏鬆"
  ]
 },
 "exaggeration": {
  "_note": "不實、誇張或易生誤解（認定準則第4條）。含無證據支持之絕對化描述。",
  "applies_to": [
   "food",
   "cosmetic",
   "general"
  ],
  "severity": "high",
  "terms": [
   "最有效",
   "最強",
   "第一名",
   "唯一",
   "獨家專利保證",
   "百分之百有效",
   "100%有效",
   "保證有效",
   "絕對有效",
   "神奇",
   "奇蹟",
   "奇效",
   "驚人效果",
   "完全無副作用",
   "零副作用",
   "無任何副作用",
   "永久",
   "終身有效",
   "一勞永逸",
   "立即見效",
   "馬上見效",
   "三天見效",
   "七天見效",
   "全球最",
   "世界第一",
   "業界唯一"
  ]
 },
 "body_function_food": {
  "_note": "涉及維持或改變人體器官、組織、生理或外觀功能之描述（認定準則第4條第3款）。未取得健康食品許可者不得宣稱。",
  "applies_to": [
   "food"
  ],
  "severity": "high",
  "terms": [
   "美白",
   "淡斑",
   "除皺",
   "抗老",
   "逆齡",
   "回春",
   "瘦身",
   "減肥",
   "燃脂",
   "消脂",
   "瘦小腹",
   "阻斷澱粉",
   "豐胸",
   "長高",
   "改善視力",
   "恢復視力",
   "治療近視",
   "預防近視",
   "增強記憶力",
   "提升智力",
   "變聰明",
   "改善腸道",
   "整腸",
   "促進排便",
   "改善便秘",
   "護眼",
   "顧眼睛"
  ]
 },
 "cosmetic_medical": {
  "_note": "化粧品不得宣稱之醫療或類醫療效能（化粧品衛生安全管理法第10條）。",
  "applies_to": [
   "cosmetic"
  ],
  "severity": "critical",
  "terms": [
   "醫學美容",
   "醫美級",
   "藥用",
   "處方級",
   "換膚",
   "去角質換膚",
   "煥膚療程",
   "治療痘痘",
   "根治粉刺",
   "消除痘疤",
   "除皺",
   "撫平皺紋",
   "填補皺紋",
   "溶脂",
   "拉皮",
   "微整形效果",
   "生髮",
   "防止掉髮",
   "治療禿頭",
   "殺菌",
   "消毒",
   "抗菌配方",
   "修復受損肌膚細胞",
   "重建肌膚屏障"
  ]
 },
 "safe_alternatives": {
  "_note": "示範用的合規替代寫法。用來說明『規則層不只是擋，還能給修改方向』。",
  "mapping": {
   "護眼": "陪伴長時間用眼的日常",
   "改善視力": "日常補充葉黃素",
   "增強免疫力": "均衡飲食之外的日常補充",
   "整腸": "順暢的一天",
   "改善便秘": "維持日常規律作息",
   "美白": "亮采、勻淨膚況（化粧品限用，食品不可）",
   "抗老": "維持日常保養節奏",
   "醫學美容": "溫和配方",
   "殺菌": "清潔",
   "立即見效": "持續使用數週後感受變化",
   "保證有效": "多數使用者回饋"
  }
 }
}

for k, v in BANNED_DATA.items():
    if isinstance(v, dict) and 'terms' in v:
        print(f"{k:24s} {len(v['terms']):>3} 詞  適用：{'/'.join(v['applies_to'])}")

## §2 Prompt v0 → v3

**設計原則：每一版只加一件事。**

這樣評測表上每一欄的改善都能歸因到單一改動。若一次加三件事，
表格會一次全綠，就失去「哪個改動帶來哪個改善」的教學價值。

| 版本 | 加了什麼 | 預期修好 |
|---|---|---|
| v0 | 什麼都不給 | — （基準線）|
| v1 | 角色／受眾／字數／必含規格 | 長度、規格覆蓋 |
| v2 | 法規禁詞 + 品牌語調 few-shot | 禁詞、語調 |
| v3 | `responseSchema` 結構化輸出 | 可機器讀取 |

In [ ]:
"""Prompt v0 → v3 演進。

設計原則：**每一版只加一件事**，這樣評測表上每一欄的改善都能歸因到單一改動。
這是 §2 的核心。

    v0  naive          什麼都不給              → 基準線
    v1  + 通路約束      角色/受眾/字數/必含規格   → 修好「長度」與「規格覆蓋」
    v2  + 法規與語調    禁詞清單 + few-shot     → 修好「禁詞」與「語調」
    v3  + 結構化輸出    responseSchema         → 修好「可機器讀取」

刻意不做的事：不在 v1 就把法規塞進去。若一次加三件事，評測表會一次全綠，
就失去「哪個改動帶來哪個改善」的教學價值。
"""

import json


# --------------------------------------------------------------------------
# v3 的輸出 schema。同時給 Gemini 當 responseSchema，也給規則層當驗證依據。
# --------------------------------------------------------------------------
COPY_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": f"電商商品標題，最多 {SHOPEE_TITLE_MAX} 字",
        },
        "bullets": {
            "type": "array",
            "items": {"type": "string"},
            "description": f"{BULLET_COUNT} 條賣點，每條最多 {BULLET_MAX} 字",
        },
        "seo_description": {
            "type": "string",
            "description": f"SEO 描述，最多 {SEO_DESC_MAX} 字",
        },
        "hashtags": {
            "type": "array",
            "items": {"type": "string"},
            "description": "3-5 個標籤，不含 # 符號",
        },
    },
    "required": ["title", "bullets", "seo_description", "hashtags"],
}


def _product_block(product: dict) -> str:
    """把商品資料整理成 prompt 用的文字區塊。"""
    specs = "\n".join(f"  - {k}：{v}" for k, v in product["specs"].items())
    return (
        f"商品名稱：{product['name']}\n"
        f"品牌：{product['brand']}\n"
        f"分類：{product['category']}\n"
        f"售價：NT${product['price']}\n"
        f"規格：\n{specs}"
    )


# --------------------------------------------------------------------------
# v0 — naive
# --------------------------------------------------------------------------
def build_v0(product: dict, **_) -> str:
    """大多數人第一次寫的 prompt。刻意保持這麼爛。"""
    return f"幫我寫這個商品的電商文案。\n\n{_product_block(product)}"


# --------------------------------------------------------------------------
# v1 — 加通路約束
# --------------------------------------------------------------------------
def build_v1(product: dict, **_) -> str:
    """加入角色、受眾、通路硬限制、必含規格。

    這一版修好的是「可上架」：長度符合平台規則、規格沒有漏。
    還沒處理法規，所以禁詞仍會出現 —— 這是刻意的。
    """
    must = "、".join(product["must_include_keywords"])
    return f"""你是台灣電商平台的資深商品文案編輯。

請為以下商品撰寫文案，目標受眾是{product['target_audience']}。

【格式要求】
- 商品標題：**不超過 {SHOPEE_TITLE_MAX} 個字**
- 賣點：{BULLET_COUNT} 條，每條**不超過 {BULLET_MAX} 個字**
- SEO 描述：**不超過 {SEO_DESC_MAX} 個字**
- 標籤：3-5 個

【必須包含的規格資訊】
以下關鍵字必須出現在文案中，不可省略或改寫：{must}

【商品資料】
{_product_block(product)}"""


# --------------------------------------------------------------------------
# v2 — 加法規約束與品牌語調
# --------------------------------------------------------------------------
def build_v2(product: dict, banned_terms: list[str], tone_examples: list[dict] = None, **_) -> str:
    """在 v1 之上加兩件事：法規禁令、品牌語調 few-shot。

    禁詞清單依商品的 regulated_category 動態帶入 —— 保健食品和 3C 適用的規則不同，
    這是規則層必須「按商品屬性套用」的原因。
    """
    base = build_v1(product)

    banned_display = "、".join(banned_terms[:60])
    legal = {
        "food": "《食品安全衛生管理法》第 28 條",
        "cosmetic": "《化粧品衛生安全管理法》第 10 條",
        "general": "《公平交易法》關於不實廣告之規範",
    }[product["regulated_category"]]

    block = f"""

【法規限制 — 這是硬性要求，違反會被開罰】
本商品受{legal}規範。文案**絕對不可**出現下列詞彙或其同義表達：
{banned_display}

具體而言，不可宣稱：
- 疾病的預防、改善、減輕、診斷或治療
- 維持或改變人體器官、組織、生理或外觀之功能
- 無證據支持的絕對化描述（最有效、保證、100%、立即見效等）

若某個賣點只能用上述詞彙表達，請改用「描述使用情境」或「描述成分事實」的方式改寫，
不要為了避開禁詞而寫出空洞的句子。

【品牌語調】
{product['brand_tone']}"""

    if tone_examples:
        samples = "\n\n".join(
            f"範例 {i + 1}（商品：{ex['product']}）\n標題：{ex['title']}\n賣點：{ex['bullet']}"
            for i, ex in enumerate(tone_examples)
        )
        block += f"""

以下是本品牌既有的合規文案，請模仿其語氣與用字習慣：

{samples}"""

    return base + block


# --------------------------------------------------------------------------
# v3 — 加結構化輸出
# --------------------------------------------------------------------------
def build_v3(product: dict, banned_terms: list[str], tone_examples: list[dict] = None, **_) -> str:
    """v2 + 結構化輸出。

    注意：prompt 本身只多一句話，真正的工作交給 API 的 responseSchema 參數
    （見 generation.py）。這是重點 —— 不要用 prompt 硬凹 JSON 格式，
    要用 API 原生的 structured output，模型才會被約束在 schema 內。
    """
    return build_v2(product, banned_terms, tone_examples) + """

【輸出格式】
請直接輸出符合指定 schema 的 JSON，不要加上 markdown 程式碼區塊標記，不要加任何說明文字。"""


PROMPT_VERSIONS = {
    "v0": build_v0,
    "v1": build_v1,
    "v2": build_v2,
    "v3": build_v3,
}

# v3 是唯一啟用 API 層 structured output 的版本
STRUCTURED_VERSIONS = {"v3"}


# --------------------------------------------------------------------------
# 品牌語調 few-shot 範例（人工撰寫，已確認合規）
# --------------------------------------------------------------------------
TONE_EXAMPLES = {
    "晨光研選": [
        {
            "product": "B群緩釋錠",
            "title": "晨光研選 緩釋B群錠 90錠 每日一錠 全素可食",
            "bullet": "8種B群一次補齊，緩釋設計",
        },
        {
            "product": "鎂錠",
            "title": "晨光研選 甘胺酸鎂錠 120錠 睡前補充 無鎮靜成分",
            "bullet": "選用好吸收的甘胺酸螯合形式",
        },
    ],
    "青研 CHINGYEN": [
        {
            "product": "神經醯胺乳液",
            "title": "青研 神經醯胺修護乳液 50ml 敏弱肌適用 無香料",
            "bullet": "三種神經醯胺，質地清爽不黏膩",
        },
        {
            "product": "溫和卸妝油",
            "title": "青研 純淨卸妝油 150ml 好沖洗 不致粉刺測試",
            "bullet": "乳化快速，沖水後不留油感",
        },
    ],
    "Volta": [
        {
            "product": "行動電源",
            "title": "Volta 20000mAh 行動電源 45W雙向快充 可上飛機",
            "bullet": "45W輸出，筆電也充得動",
        },
        {
            "product": "USB-C 傳輸線",
            "title": "Volta 240W USB-C 編織線 2M 支援40Gbps傳輸",
            "bullet": "240W過電，8K螢幕直出",
        },
    ],
    "溯源焙所": [
        {
            "product": "日曬西達摩掛耳",
            "title": "溯源焙所 日曬西達摩掛耳咖啡 10入 中淺焙",
            "bullet": "日曬處理，草莓與黑糖尾韻",
        },
        {
            "product": "阿里山烏龍",
            "title": "溯源焙所 阿里山高山烏龍 三角茶包 20入 海拔1400m",
            "bullet": "清香型輕發酵，冷泡熱泡皆宜",
        },
    ],
}


def get_banned_terms_for(product: dict, banned_data: dict) -> list[str]:
    """依商品的 regulated_category 取出適用的禁詞。

    保健食品要擋『護眼』，3C 不用 —— 規則必須跟著商品屬性走，
    這是規則層在真實系統裡最容易被做錯的地方。
    """
    category = product["regulated_category"]
    terms: list[str] = []
    for group in banned_data.values():
        if not isinstance(group, dict) or "terms" not in group:
            continue
        if category in group.get("applies_to", []):
            terms.extend(group["terms"])
    return sorted(set(terms))

### 看一下四個版本的實際差異

In [ ]:
p = PRODUCTS[0]   # 葉黃素 — 保健食品，法規風險最高的類別
terms = get_banned_terms_for(p, BANNED_DATA)

for name, fn in PROMPT_VERSIONS.items():
    text = fn(p, banned_terms=terms, tone_examples=TONE_EXAMPLES[p['brand']])
    print(f"{name}: {len(text):>5,} 字元")

print("\n" + "=" * 72)
print("v0 全文（這就是大多數人第一次寫的 prompt）")
print("=" * 72)
print(PROMPT_VERSIONS['v0'](p))

In [ ]:
print("=" * 72)
print("v3 全文 — 注意法規段落與 few-shot 是怎麼寫的")
print("=" * 72)
print(PROMPT_VERSIONS['v3'](p, banned_terms=terms, tone_examples=TONE_EXAMPLES[p['brand']]))

### 生成

`generate(..., structured=True)` 時啟用 API 原生的 `responseSchema` ——
**這是 v3 與 v2 的唯一差別**（呼叫層的程式碼在 §0）。

**重點：不要用 prompt 硬凹 JSON 格式，要用 API 參數約束。**
prompt 裡寫「請輸出 JSON」模型會照做九成的時間；用 `responseSchema`
模型是被解碼器約束在 schema 內，那一成的失敗才會消失。

In [ ]:
ledger = UsageLedger()
outputs = {}   # (sku, version) -> 文案原文

t0 = time.time()
for i, product in enumerate(PRODUCTS, 1):
    terms = get_banned_terms_for(product, BANNED_DATA)
    examples = TONE_EXAMPLES.get(product['brand'])
    for version, builder in PROMPT_VERSIONS.items():
        prompt = builder(product, banned_terms=terms, tone_examples=examples)
        res = generate(client, prompt, structured=(version in STRUCTURED_VERSIONS))
        ledger.record(f"gen:{version}", res.usage)
        outputs[(product['sku'], version)] = res.text
        if res.error:
            print(f"  ⚠ {product['sku']} {version}: {res.error}")
    print(f"[{i:>2}/{len(PRODUCTS)}] {product['sku']} 完成")

print(f"\n共 {len(outputs)} 筆，耗時 {time.time() - t0:.1f} 秒")

### 肉眼比對：同一個商品，v0 vs v3

In [ ]:
sku = PRODUCTS[0]['sku']
print("=" * 72); print(f"{sku}  v0"); print("=" * 72)
print(outputs[(sku, 'v0')])
print("\n" + "=" * 72); print(f"{sku}  v3"); print("=" * 72)
print(outputs[(sku, 'v3')])

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **33:30 — ▶ 重跑 v0。**「看起來不錯對吧？文筆很好。」（停頓）「但它不能用。」
>
> **35:00 — ▶ 重跑 v3。**「同一個商品。」
>
> 先不要解釋為什麼 v3 比較好 —— 讓他們自己看。解釋留到對比表。

</div>

## §3 評測 — 這一節是重點

三層評測，由便宜到貴：

1. **規則層** — 免費、毫秒級、確定性，可以進 CI
2. **LLM judge** — 有成本、有偏誤，只評規則層評不了的東西
3. **人工抽樣** — 最貴，只看前兩層有分歧的

**順序很重要。** 多數團隊一上來就做 LLM judge，又貴又不穩。
先建規則層：它擋掉大部分明確錯誤，而且結果可重現。

### 第一層：規則

注意 `parse_freeform()` —— 為了評測 v0～v2 的自由文字輸出，
我們被迫寫一個靠正則猜測的 parser。**模型換個排版它就壞掉。**

這段脆弱的程式碼本身就是「為什麼要用 structured output」最好的論據。
v3 之後這整段可以刪掉。

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **36:30 — 秀 `parse_freeform()`，這段工程師最有共鳴。**
>
> 「為了評 v0，我得寫這種靠正則猜測的 parser。模型換個排版它就壞掉。
> **v3 之後這段可以整個刪掉。**」

</div>

In [ ]:
"""評測第一層：規則層。

不呼叫任何模型，毫秒級，零成本。重點：**先建這一層**。
多數團隊一上來就做 LLM judge，又貴又不穩；規則層能擋掉大部分明確錯誤，
而且結果是確定性的 —— 同樣的輸入永遠得到同樣的判定，可以進 CI。

這個模組刻意不 import 任何 GCP 套件，因此離線可測、可單元測試。
"""

from __future__ import annotations

import json
import re
from dataclasses import dataclass, field



# --------------------------------------------------------------------------
# 把模型輸出正規化成統一結構
# --------------------------------------------------------------------------
@dataclass
class ParsedCopy:
    """從模型輸出解析出的文案結構。"""

    title: str = ""
    bullets: list[str] = field(default_factory=list)
    seo_description: str = ""
    hashtags: list[str] = field(default_factory=list)
    raw: str = ""
    is_structured: bool = False  # True = 直接來自合法 JSON，不需猜測


_TITLE_PAT = re.compile(
    r"^\s*(?:\**)\s*(?:商品)?標題\s*(?:\**)\s*[:：]\s*(?:\**)\s*(.+?)\s*(?:\**)\s*$",
    re.MULTILINE,
)
_BULLET_PAT = re.compile(r"^\s*(?:[-*•・‧]|\d+[.、)])\s*(.+?)\s*$", re.MULTILINE)
_SEO_PAT = re.compile(
    r"^\s*(?:\**)\s*(?:SEO\s*)?(?:描述|說明)\s*(?:\**)\s*[:：]\s*(?:\**)\s*(.+?)\s*(?:\**)\s*$",
    re.MULTILINE | re.IGNORECASE,
)
_HASHTAG_PAT = re.compile(r"#([^\s#,，、]+)")
_FENCE_PAT = re.compile(r"^\s*```(?:json)?\s*|\s*```\s*$", re.MULTILINE)


def parse_structured(text: str) -> ParsedCopy | None:
    """嘗試把輸出當成 JSON 解析。成功才算 schema 合法。"""
    cleaned = _FENCE_PAT.sub("", text).strip()
    try:
        data = json.loads(cleaned)
    except (json.JSONDecodeError, ValueError):
        return None
    if not isinstance(data, dict):
        return None

    required = {"title", "bullets", "seo_description", "hashtags"}
    if not required.issubset(data.keys()):
        return None
    if not isinstance(data["bullets"], list) or not isinstance(data["hashtags"], list):
        return None
    if not isinstance(data["title"], str) or not isinstance(data["seo_description"], str):
        return None

    return ParsedCopy(
        title=data["title"].strip(),
        bullets=[str(b).strip() for b in data["bullets"]],
        seo_description=data["seo_description"].strip(),
        hashtags=[str(h).lstrip("#").strip() for h in data["hashtags"]],
        raw=text,
        is_structured=True,
    )


def parse_freeform(text: str) -> ParsedCopy:
    """從自由文字裡「猜」出結構。

    ⚠️ 這個函式是脆弱的，而那正是重點。
    為了評測 v0～v2 的輸出，我們被迫寫這種靠正則猜測的 parser：
    模型換個排版它就壞掉。v3 用 responseSchema 之後這段就可以整個刪掉。
    這段脆弱的程式碼本身，就是「為什麼要 structured output」最好的論據。
    """
    title_match = _TITLE_PAT.search(text)
    if title_match:
        title = title_match.group(1)
    else:
        # 退而求其次：第一行非空、非 markdown 標記的文字
        title = ""
        for line in text.splitlines():
            stripped = line.strip().lstrip("#").strip()
            if stripped and not stripped.startswith("```"):
                title = stripped
                break

    bullets = [b for b in _BULLET_PAT.findall(text) if len(b) > 2]

    seo_match = _SEO_PAT.search(text)
    seo = seo_match.group(1) if seo_match else ""

    hashtags = _HASHTAG_PAT.findall(text)

    return ParsedCopy(
        title=title.strip(),
        bullets=[b.strip() for b in bullets],
        seo_description=seo.strip(),
        hashtags=hashtags,
        raw=text,
        is_structured=False,
    )


def parse_output(text: str) -> ParsedCopy:
    """先試結構化，失敗才退回猜測。"""
    return parse_structured(text) or parse_freeform(text)


# --------------------------------------------------------------------------
# 規則檢查
# --------------------------------------------------------------------------
@dataclass
class RuleResult:
    sku: str
    version: str
    schema_valid: bool
    title_length_ok: bool
    title_length: int
    bullet_length_ok: bool
    seo_length_ok: bool
    spec_coverage: float          # 0.0 - 1.0
    spec_missing: list[str]
    banned_clean: bool
    banned_hits: list[str]

    @property
    def all_pass(self) -> bool:
        return (
            self.schema_valid
            and self.title_length_ok
            and self.bullet_length_ok
            and self.seo_length_ok
            and self.spec_coverage == 1.0
            and self.banned_clean
        )


def _searchable_text(parsed: ParsedCopy) -> str:
    """組出用來搜尋禁詞與規格關鍵字的文字範圍。

    這裡有一個容易做錯、而且錯了會出事的決定：**掃描範圍要多大？**

    - 結構化輸出（v3）：掃 parsed 後的欄位就夠，範圍精確，不會把 JSON key
      名稱或模型的客套話算進去。
    - 自由文字（v0～v2）：**必須掃 raw 全文**。因為脆弱的 parser 常常只抓到
      第一行，真正的違規詞往往躲在後面幾段。只掃 parsed 欄位會讓違規漏網 ——
      而漏檢比誤判危險得多（誤判只是多花人力複查，漏檢是收罰單）。

    這條差異本身就是「為什麼要用 structured output」的另一個論據：
    有 schema 才能精確掃描，沒有 schema 就只能全文粗掃、誤判率上升。
    """
    if parsed.is_structured:
        return " ".join(
            [parsed.title, *parsed.bullets, parsed.seo_description, *parsed.hashtags]
        ).strip()
    return parsed.raw


def check_banned(text: str, banned_terms: list[str]) -> list[str]:
    """回傳所有命中的禁詞。

    用長詞優先比對，避免「治療」和「治療過敏」重複計數同一段文字。
    """
    hits: list[str] = []
    consumed: list[tuple[int, int]] = []
    for term in sorted(banned_terms, key=len, reverse=True):
        for m in re.finditer(re.escape(term), text):
            span = (m.start(), m.end())
            if any(span[0] >= s and span[1] <= e for s, e in consumed):
                continue  # 已被更長的禁詞涵蓋
            consumed.append(span)
            hits.append(term)
            break  # 同一個詞只記一次
    return hits


def evaluate_rules(
    output_text: str,
    product: dict,
    version: str,
    banned_terms: list[str],
) -> RuleResult:
    """對單一筆模型輸出跑完整規則層。"""
    parsed = parse_output(output_text)
    text = _searchable_text(parsed)

    missing = [kw for kw in product["must_include_keywords"] if kw not in text]
    coverage = 1.0 - len(missing) / max(len(product["must_include_keywords"]), 1)

    bullets_ok = bool(parsed.bullets) and all(
        len(b) <= BULLET_MAX for b in parsed.bullets
    )
    seo_ok = 0 < len(parsed.seo_description) <= SEO_DESC_MAX

    hits = check_banned(text, banned_terms)

    return RuleResult(
        sku=product["sku"],
        version=version,
        schema_valid=parsed.is_structured,
        title_length_ok=0 < len(parsed.title) <= SHOPEE_TITLE_MAX,
        title_length=len(parsed.title),
        bullet_length_ok=bullets_ok,
        seo_length_ok=seo_ok,
        spec_coverage=round(coverage, 3),
        spec_missing=missing,
        banned_clean=not hits,
        banned_hits=hits,
    )


def suggest_fix(hits: list[str], banned_data: dict) -> dict[str, str]:
    """對命中的禁詞給出合規替代寫法。

    重點：規則層不只是「擋」，還能「給修改方向」。
    這讓它從一個惹人厭的 linter 變成一個有用的工具。
    """
    mapping = banned_data.get("safe_alternatives", {}).get("mapping", {})
    return {h: mapping[h] for h in hits if h in mapping}

In [ ]:
"""把評測結果整理成「版本 × 指標」對比表。

這張表是整份分析的結論。前面所有步驟都是為了讓它可信。

設計原則：
- 每一欄對應一個 prompt 改動所修好的問題，欄位順序＝改動順序，階梯感才會出來。
- 顯示通過率（%）而非平均分數，因為「12 個商品裡有幾個能直接上架」比
  「平均 3.8 分」更接近商業決策者要的答案。
"""

from __future__ import annotations


METRIC_LABELS = {
    "schema_valid": "可機器讀取",
    "title_length_ok": "標題長度合規",
    "bullet_length_ok": "賣點長度合規",
    "seo_length_ok": "SEO描述合規",
    "spec_full": "規格完整覆蓋",
    "banned_clean": "法規禁詞 0 命中",
}

# 每個指標是被哪一版的改動修好的 —— 用於標註歸因
FIXED_BY = {
    "title_length_ok": "v1",
    "bullet_length_ok": "v1",
    "seo_length_ok": "v1",
    "spec_full": "v1",
    "banned_clean": "v2",
    "schema_valid": "v3",
}


def _metric_values(r: RuleResult) -> dict[str, bool]:
    return {
        "schema_valid": r.schema_valid,
        "title_length_ok": r.title_length_ok,
        "bullet_length_ok": r.bullet_length_ok,
        "seo_length_ok": r.seo_length_ok,
        "spec_full": r.spec_coverage == 1.0,
        "banned_clean": r.banned_clean,
    }


def build_table(results: list[RuleResult]) -> dict[str, dict[str, float]]:
    """results 為所有 (商品 × 版本) 的規則結果，回傳 {版本: {指標: 通過率}}。"""
    by_version: dict[str, list[RuleResult]] = {}
    for r in results:
        by_version.setdefault(r.version, []).append(r)

    table: dict[str, dict[str, float]] = {}
    for version in sorted(by_version):
        rows = by_version[version]
        agg = {m: 0 for m in METRIC_LABELS}
        for r in rows:
            for m, ok in _metric_values(r).items():
                agg[m] += int(ok)
        table[version] = {m: round(100 * c / len(rows), 1) for m, c in agg.items()}
        table[version]["_全數通過"] = round(
            100 * sum(r.all_pass for r in rows) / len(rows), 1
        )
    return table


def to_dataframe(results: list[RuleResult]):
    """回傳 pandas DataFrame，notebook 裡直接顯示用。"""
    import pandas as pd

    table = build_table(results)
    ordered = [
        "title_length_ok",
        "bullet_length_ok",
        "seo_length_ok",
        "spec_full",
        "banned_clean",
        "schema_valid",
    ]
    df = pd.DataFrame(
        {
            METRIC_LABELS[m] + f"\n(v{FIXED_BY[m][1:]}修好)": [
                table[v][m] for v in sorted(table)
            ]
            for m in ordered
        },
        index=sorted(table),
    )
    df["★ 全數通過"] = [table[v]["_全數通過"] for v in sorted(table)]
    df.index.name = "Prompt 版本"
    return df


def render_text_table(results: list[RuleResult]) -> str:
    """純文字版對比表 —— 不依賴 pandas，離線也能看。"""
    table = build_table(results)
    ordered = [
        "title_length_ok",
        "bullet_length_ok",
        "seo_length_ok",
        "spec_full",
        "banned_clean",
        "schema_valid",
    ]
    versions = sorted(table)

    def pad_to(text: str, width: int) -> str:
        """CJK 字元在等寬終端佔兩格，用顯示寬度而非字元數對齊。"""
        display = sum(2 if ord(c) > 127 else 1 for c in text)
        return text + " " * max(width - display, 1)

    width = 22 + 12 + 8 * len(versions)
    lines = [
        pad_to("指標", 22) + pad_to("修好的版本", 12) + "".join(f"{v:>8}" for v in versions),
        "─" * width,
    ]

    for m in ordered:
        lines.append(
            pad_to(METRIC_LABELS[m], 22)
            + pad_to(FIXED_BY[m], 12)
            + "".join(f"{table[v][m]:>7.0f}%" for v in versions)
        )

    lines.append("─" * width)
    lines.append(
        pad_to("★ 全數通過", 22)
        + pad_to("", 12)
        + "".join(f"{table[v]['_全數通過']:>7.0f}%" for v in versions)
    )
    return "\n".join(lines)


def noise_floor(n_items: int, n_checks_each: int = 1) -> float:
    """這張表上「多少百分點以內的差距應該當成雜訊」。

    用最粗的估計：一次檢查翻轉所造成的百分點變化。12 個商品 × 7 條 rubric
    = 84 次檢查，翻一條就是 1.2 個百分點；所以 2 個百分點以內的差距
    不該當成改善。

    為什麼要放這個函式：demo 的樣本數很小（12 個商品），
    很容易看到 92.9% vs 95.2% 就宣稱「v2 比較好」，但那其實只差一次檢查。
    **把雜訊當成訊號，是評測工作最容易犯、也最傷的錯。**
    """
    total = max(n_items * n_checks_each, 1)
    return round(100.0 / total, 1)


def significance_note(n_items: int, n_checks_each: int = 1) -> str:
    floor = noise_floor(n_items, n_checks_each)
    return (
        f"⚠ 樣本數 {n_items} 筆 × 每筆 {n_checks_each} 項 = {n_items * n_checks_each} 次檢查。\n"
        f"  單次檢查翻轉 ≈ {floor} 個百分點。\n"
        f"  **差距在 {floor * 2:.0f} 個百分點以內時，請當成雜訊，不要宣稱是改善。**\n"
        f"  要做出可靠的小幅比較，需要更大的評測集（實務上 200 筆以上）。"
    )


def combined_table(rule_results: list[RuleResult], rubric_summary: dict):
    """規則層 + rubric 層的合併對比表。

    每一版只負責修好一件事，這張表要能讓那件事一眼看出來。
    """
    import pandas as pd

    table = build_table(rule_results)
    versions = sorted(table)
    rows = {
        "標題長度合規 (v1)": [table[v]["title_length_ok"] for v in versions],
        "規格完整覆蓋 (v1)": [table[v]["spec_full"] for v in versions],
        "法規禁詞 0 命中 (v1)": [table[v]["banned_clean"] for v in versions],
        "rubric 通過率 (v2)": [
            rubric_summary.get(v, {}).get("rubric通過率%", float("nan")) for v in versions
        ],
        "可直接上架 (v2)": [
            rubric_summary.get(v, {}).get("可直接上架%", float("nan")) for v in versions
        ],
        "可機器讀取 (v3)": [table[v]["schema_valid"] for v in versions],
    }
    df = pd.DataFrame(rows, index=versions)
    df.index.name = "Prompt 版本"
    return df


def violation_detail(results: list[RuleResult], version: str, limit: int = 5) -> str:
    """列出某一版的違規細節 —— demo 中用來證明『這不是假資料』。"""
    rows = [r for r in results if r.version == version and not r.banned_clean]
    if not rows:
        return f"{version}：無禁詞違規 ✓"
    out = [f"{version} 的禁詞命中（前 {limit} 筆）："]
    for r in rows[:limit]:
        out.append(f"  {r.sku}  →  {'、'.join(r.banned_hits)}")
    if len(rows) > limit:
        out.append(f"  ...另有 {len(rows) - limit} 筆")
    return "\n".join(out)

### ★ 規則層對比表

In [ ]:
rule_results = []
for product in PRODUCTS:
    terms = get_banned_terms_for(product, BANNED_DATA)
    for version in PROMPT_VERSIONS:
        rule_results.append(
            evaluate_rules(outputs[(product['sku'], version)], product, version, terms)
        )

print(render_text_table(rule_results))

In [ ]:
# 同一張表的 DataFrame 版本（投影用，數字大一點好讀）
to_dataframe(rule_results).style.format("{:.0f}%").background_gradient(cmap="RdYlGn", vmin=0, vmax=100)

### 違規細節 — 證明這不是假資料

規則層不只能擋，還能給修改方向。這讓它從一個惹人厭的 linter
變成文案人員真的願意用的工具。

In [ ]:
for version in ['v0', 'v1', 'v2', 'v3']:
    print(violation_detail(rule_results, version))
    print()

# 對命中的禁詞給出合規替代寫法
worst = max((r for r in rule_results if r.version == 'v0'),
            key=lambda r: len(r.banned_hits), default=None)
if worst and worst.banned_hits:
    print("=" * 60)
    print(f"{worst.sku} 的修改建議：")
    for bad, good in suggest_fix(worst.banned_hits, BANNED_DATA).items():
        print(f"  ✗ {bad}  →  ✓ {good}")

### 第二層：LLM-as-judge —— 用二元 rubric，不用 1–5 分

只評規則層評不了的：語調、賣點覆蓋。

#### 為什麼不用 1–5 分？

早期的 LLM-as-judge 幾乎都用 Likert 量表。實務上它有幾個難以修復的問題：

| 問題 | 後果 |
|---|---|
| 分數擠在 3–4 分 | 鑑別度低，看不出差異 |
| 同一份文案今天 4 分明天 3 分 | 不可重現，無法判斷是模型變差還是評審漂移 |
| 換評審模型整組分數平移 | 歷史數據作廢 |
| 寫得長、寫得華麗容易拿高分 | verbosity bias |
| **「3.8 分」無法行動** | 你不知道要改什麼 |

二元 rubric 把一個模糊的大問題，拆成一組**具體、可檢查的 yes/no 小問題**：

```
✗  「這份文案的品質有幾分？」          → 3.8 / 5，然後呢？
✓  「文案是否寫出游離型葉黃素 30mg？」  → 否 → 知道要補什麼
```

這也是 Vertex AI Gen AI Evaluation Service 把 adaptive rubrics
形容成**「像單元測試」**的原因。

#### 公平性：rubric 只從商品資料生成，不看被評的文案

**每個商品產生一次，v0～v3 共用同一組。**
如果讓評審看著文案即興出題，每個版本會被問不同的問題 ——
那等於每個考生考不同的考卷，對比表就沒有意義了。

In [ ]:
"""評測第二層：LLM-as-judge，採用**二元自適應 rubric**（pass / fail 檢查清單）。

只評「規則層評不了」的東西 —— 語調、賣點覆蓋這類需要語意理解的維度。
順序很重要：**先跑規則層，被擋下來的不必送 judge**，省下的是真金白銀。

---

## 為什麼不用 1–5 分？

這是這個模組最重要的設計決定。

早期的 LLM-as-judge 幾乎都用 Likert 量表（1–5 分）。實務上它有幾個難以修復的問題：

- **分數擠在中間。** 模型很少給 1 或 5，結果多數樣本都是 3–4 分，鑑別度低。
- **不可重現。** 同一份文案今天 4 分明天 3 分，你無法判斷是模型變差還是評審漂移。
- **跨模型不一致。** 換一個評審模型，整組分數就平移，歷史數據全部作廢。
- **容易被長度騙。** 寫得長、寫得華麗容易拿高分，這正是 verbosity bias。
- **「3.8 分」無法行動。** 你不知道要改什麼。

二元 rubric 把一個模糊的大問題拆成一組**具體、可檢查的 yes/no 小問題**：

    ✗  「這份文案的品質有幾分？」        → 3.8 / 5，然後呢？
    ✓  「文案是否提到 30mg 這個含量？」  → 否 → 知道要補什麼

好處是直接的：**判準明確、可重現、可累積、能直接對應到修改動作**，
而且 pass/fail 很難用堆字數來灌水。這也是為什麼 Vertex AI 的
Gen AI Evaluation Service 把 adaptive rubrics 形容成「像單元測試」。

> 註：Vertex AI 有託管版本的 rubric 評測。本模組選擇手寫，原因有二：
> 一是要看得到 rubric 長什麼樣子，那才是可帶走的概念；
> 二是託管服務的 SDK 介面仍在變動（本機安裝的 `vertexai.evaluation`
> 只有舊的 Likert 型 PointwiseMetric，新的 adaptive rubric 走另一個介面）。
> 概念完全相同，要換成託管版本時，換的是呼叫方式，不是方法論。

---

## 公平性：rubric 只從商品資料生成，不看被評的文案

這點很容易做錯，做錯了整張對比表就沒有意義。

rubric **對每個商品產生一次，v0～v3 共用同一組**。
如果讓評審看著文案即興出題，每個版本會被問不同的問題，
分數就不可比 —— 那等於每個考生考不同的考卷。
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field


# --------------------------------------------------------------------------
# 已知偏誤與本專案的緩解方式
# --------------------------------------------------------------------------
JUDGE_BIASES = {
    "self-preference": {
        "說明": "模型傾向給自己產出的內容較高分。",
        "緩解": f"評審模型（{JUDGE_MODEL}）與生成模型（{GEN_MODEL}）刻意選不同的。",
    },
    "verbosity": {
        "說明": "較長、較華麗的回答容易被評為較好，即使內容沒有更好。",
        "緩解": "改用 pass/fail 判準，「有沒有提到 30mg」無法靠寫得長來灌水。",
    },
    "position": {
        "說明": "成對比較時，先出現的選項容易獲勝。",
        "緩解": "採 pointwise（單篇對照 rubric），不做 pairwise，從源頭避開。",
    },
    "score-clustering": {
        "說明": "Likert 分數容易全部擠在 3–4 分，鑑別度低且不可重現。",
        "緩解": "不用分數。二元判準沒有中間地帶可以躲。",
    },
    "criteria-drift": {
        "說明": "評審每次自己想判準，等於每個版本考不同的考卷。",
        "緩解": "rubric 只從商品資料生成一次，v0～v3 共用同一組。",
    },
}

# --------------------------------------------------------------------------
# Schemas
# --------------------------------------------------------------------------
RUBRIC_GEN_SCHEMA = {
    "type": "object",
    "properties": {
        "rubrics": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "string", "description": "R1, R2, ..."},
                    "criterion": {
                        "type": "string",
                        "description": "一句可用是／否回答的具體判準",
                    },
                    "dimension": {
                        "type": "string",
                        "enum": ["賣點覆蓋", "品牌語調", "消費者可讀性"],
                    },
                    "critical": {
                        "type": "boolean",
                        "description": "未通過就不該上架者為 true",
                    },
                },
                "required": ["id", "criterion", "dimension", "critical"],
            },
        }
    },
    "required": ["rubrics"],
}

RUBRIC_CHECK_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "string"},
                    "passed": {"type": "boolean"},
                    "evidence": {
                        "type": "string",
                        "description": "通過就引用文案中的原文；未通過就說明缺什麼",
                    },
                },
                "required": ["id", "passed", "evidence"],
            },
        }
    },
    "required": ["results"],
}

# --------------------------------------------------------------------------
# Prompts
# --------------------------------------------------------------------------
RUBRIC_GEN_PROMPT = """你是資深電商文案主管，要為一個商品訂出「文案驗收清單」。

這份清單會用來檢查 AI 產出的文案能不能上架。請產生 7 條判準。

【⚠️ 最重要：字數預算】
被檢查的文案有嚴格的通路字數限制，總共只有約 {total_budget} 個字：

  - 標題：最多 {title_max} 字
  - 賣點：{bullet_count} 條，每條最多 {bullet_max} 字
  - SEO 描述：最多 {seo_max} 字

**你的每一條判準都必須是在這個字數預算內可以達成的。**

這是最容易出錯的地方。不要寫出需要長篇說明才能滿足的判準：

  ✗ 「文案是否說明為何隨餐食用有助吸收？」    （解釋機制，30 字的賣點塞不下）
  ✗ 「文案是否說明黃金比例能提供更全面的保護？」（同上）
  ✓ 「文案是否寫出葉黃素 30mg 的含量？」       （幾個字就能達成）
  ✓ 「文案是否點出適合長時間看螢幕的族群？」    （一句話能達成）

**如果一條判準只有把文案寫長才可能通過，它就是壞判準。**
壞判準會讓冗長雜亂的文案得高分、讓精簡合規的文案被扣分，
整張評測表的結論就會顛倒。

【判準的寫法】
- 每一條都必須能用「是」或「否」回答，不可以是程度問題。
  ✗ 「文案的語氣是否夠專業？」（程度問題，無法一致判定）
  ✓ 「文案是否避免使用驚嘆號？」（可以直接看出來）
- 判準要**具體指向這個商品**，不要寫成通用的空話。

【不要納入的範圍】
以下已由程式規則自動檢查，**不要**寫成判準：字數是否超標、JSON 格式是否合法、
法規禁詞、必含關鍵字是否存在。你只負責語意層面。

【維度分配 — 必須照這個比例，不可偏重賣點覆蓋】
- 賣點覆蓋：3 條（把規格轉成消費者看得懂的好處，但要在字數內可達成）
- 品牌語調：2 條（是否符合下面的品牌語調設定）
- 消費者可讀性：2 條（是否精簡、無冗詞、無誇大語氣、可直接上架）

【critical 的判定】
未通過就不該上架的設為 true，請設 2 條為 critical。

【商品資料】
商品名稱：{name}
品牌：{brand}
品牌語調設定：{brand_tone}
目標受眾：{audience}
規格：
{specs}

請依 schema 輸出判準清單。"""

RUBRIC_CHECK_PROMPT = """你是電商文案審核員。請對照驗收清單逐條檢查下面這份文案。

【文案的字數預算】
這份文案受通路限制，總共只有約 {total_budget} 個字可用
（標題 {title_max} 字、{bullet_count} 條賣點各 {bullet_max} 字、SEO 描述 {seo_max} 字）。

**請在這個前提下判斷。** 文案簡潔不是缺點，那是通路要求。
不要因為文案沒有展開說明、沒有解釋原理就判定不通過 ——
只要在字數內把該講的講到了，就算通過。

【檢查原則】
- 每一條**只回答通過或不通過**，不要給分數。
- 判斷依據只能是文案中實際出現的內容，不要腦補。
- 通過時，`evidence` 請引用文案中的原文片段。
- 不通過時，`evidence` 請說明缺少什麼。
- **絕對不要因為文案較長、較華麗就傾向通過。**
  冗長、堆砌形容詞、重複同一個賣點，在「消費者可讀性」維度應判為不通過。
- 文案可能是自由文字，也可能是 JSON，兩者一視同仁看內容。

【驗收清單】
{rubric_list}

【待檢查的文案】
{copy_text}

請依 schema 逐條回覆，results 的長度必須與清單條數相同。"""


# --------------------------------------------------------------------------
# 資料結構
# --------------------------------------------------------------------------
@dataclass
class Rubric:
    id: str
    criterion: str
    dimension: str
    critical: bool


@dataclass
class RubricSet:
    sku: str
    rubrics: list[Rubric] = field(default_factory=list)
    error: str | None = None

    def as_prompt_list(self) -> str:
        return "\n".join(
            f"{r.id}. [{r.dimension}]{'（critical）' if r.critical else ''} {r.criterion}"
            for r in self.rubrics
        )


@dataclass
class RubricReport:
    sku: str
    version: str
    passed_ids: list[str] = field(default_factory=list)
    failed: list[tuple[str, str, bool]] = field(default_factory=list)  # (id, 原因, critical)
    total: int = 0
    error: str | None = None

    @property
    def pass_rate(self) -> float:
        return round(100 * len(self.passed_ids) / self.total, 1) if self.total else 0.0

    @property
    def critical_failures(self) -> int:
        return sum(1 for _, _, crit in self.failed if crit)

    @property
    def would_publish(self) -> bool:
        """critical 全過，且整體通過率達 80% 才算能上架。"""
        return self.total > 0 and self.critical_failures == 0 and self.pass_rate >= 80.0


# --------------------------------------------------------------------------
# 呼叫
# --------------------------------------------------------------------------
def _call_json(client, prompt: str, schema: dict, tag: str, ledger=None):
    """走 structured output 呼叫評審模型並解析 JSON。

    評審本身也用 responseSchema —— 解析失敗會讓整張評測表出現空洞，
    比生成失敗更難察覺。
    """
    import time

    from google.genai import types


    started = time.time()
    resp = client.models.generate_content(
        model=JUDGE_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=JUDGE_TEMPERATURE,
            response_mime_type="application/json",
            response_schema=schema,
        ),
    )
    meta = resp.usage_metadata
    if ledger is not None:
        ledger.record(
            tag,
            Usage(
                model=JUDGE_MODEL,
                input_tokens=getattr(meta, "prompt_token_count", 0) or 0,
                output_tokens=getattr(meta, "candidates_token_count", 0) or 0,
                latency_s=round(time.time() - started, 2),
            ),
        )
    return json.loads(resp.text)


def _budget() -> dict:
    """文案的字數預算。

    rubric 生成與檢查都必須知道這個數字，否則會產生「解釋原理」這種
    在 30 字賣點裡不可能達成的判準 —— 那會讓冗長的文案得高分、
    精簡合規的文案被扣分，整張評測表的結論顛倒過來。
    """
    total = (
        SHOPEE_TITLE_MAX
        + BULLET_COUNT * BULLET_MAX
        + SEO_DESC_MAX
    )
    return {
        "title_max": SHOPEE_TITLE_MAX,
        "bullet_count": BULLET_COUNT,
        "bullet_max": BULLET_MAX,
        "seo_max": SEO_DESC_MAX,
        "total_budget": total,
    }


def generate_rubrics(client, product: dict, ledger=None) -> RubricSet:
    """為單一商品產生驗收清單。

    ⚠️ 每個商品只做一次，v0～v3 共用。不要對每個版本重新生成 —— 那會讓
    每個版本被問不同的問題，對比表就失去意義。
    """
    specs = "\n".join(f"  - {k}：{v}" for k, v in product["specs"].items())
    prompt = RUBRIC_GEN_PROMPT.format(
        name=product["name"],
        brand=product["brand"],
        brand_tone=product["brand_tone"],
        audience=product["target_audience"],
        specs=specs,
        **_budget(),
    )
    try:
        data = _call_json(client, prompt, RUBRIC_GEN_SCHEMA, "judge:rubric_gen", ledger)
        return RubricSet(
            sku=product["sku"],
            rubrics=[
                Rubric(
                    id=str(r["id"]),
                    criterion=str(r["criterion"]),
                    dimension=str(r["dimension"]),
                    critical=bool(r["critical"]),
                )
                for r in data["rubrics"]
            ],
        )
    except Exception as e:  # noqa: BLE001
        return RubricSet(sku=product["sku"], error=str(e))


def check_against_rubrics(
    client,
    copy_text: str,
    rubric_set: RubricSet,
    version: str,
    ledger=None,
) -> RubricReport:
    """逐條檢查文案。"""
    if rubric_set.error or not rubric_set.rubrics:
        return RubricReport(
            sku=rubric_set.sku,
            version=version,
            error=rubric_set.error or "沒有可用的 rubric",
        )

    by_id = {r.id: r for r in rubric_set.rubrics}
    prompt = RUBRIC_CHECK_PROMPT.format(
        rubric_list=rubric_set.as_prompt_list(),
        copy_text=copy_text[:4000],  # 防止異常長輸出把成本吃掉
        **_budget(),
    )
    try:
        data = _call_json(client, prompt, RUBRIC_CHECK_SCHEMA, f"judge:{version}", ledger)
        passed, failed = [], []
        for row in data["results"]:
            rid = str(row["id"])
            rub = by_id.get(rid)
            if rub is None:
                continue  # 評審捏造了不存在的 id，忽略
            if bool(row["passed"]):
                passed.append(rid)
            else:
                failed.append((rid, str(row.get("evidence", "")), rub.critical))
        return RubricReport(
            sku=rubric_set.sku,
            version=version,
            passed_ids=passed,
            failed=failed,
            total=len(by_id),
        )
    except Exception as e:  # noqa: BLE001
        return RubricReport(sku=rubric_set.sku, version=version, error=str(e))


def should_judge(rule_result) -> bool:
    """規則層已經擋下的，就不要送 judge —— 這是省錢的關鍵。

    語意檢查對一份已經違反法規的文案沒有意義，而每一次 judge 呼叫
    都是用比生成更貴的模型在燒錢。
    """
    return rule_result.banned_clean and rule_result.spec_coverage > 0.5


def summarize_rubrics(reports: list[RubricReport]) -> dict[str, dict[str, float]]:
    """彙整成 {版本: 指標}，供對比表使用。"""
    by_version: dict[str, list[RubricReport]] = {}
    for r in reports:
        if r.error:
            continue
        by_version.setdefault(r.version, []).append(r)

    out = {}
    for version, rows in sorted(by_version.items()):
        if not rows:
            continue
        out[version] = {
            "rubric通過率%": round(sum(r.pass_rate for r in rows) / len(rows), 1),
            "critical違反數": sum(r.critical_failures for r in rows),
            "可直接上架%": round(100 * sum(r.would_publish for r in rows) / len(rows), 1),
            "受評筆數": len(rows),
        }
    return out


def most_common_failures(reports: list[RubricReport], rubric_sets: dict, limit: int = 5):
    """哪些判準最常沒過 —— 這才是能拿去改 prompt 的資訊。

    這正是二元 rubric 勝過分數的地方：「3.8 分」不能行動，
    「有 9 個商品沒寫出含量」可以。
    """
    counts: dict[tuple[str, str], int] = {}
    for rep in reports:
        if rep.error:
            continue
        rs = rubric_sets.get(rep.sku)
        if not rs:
            continue
        text_by_id = {r.id: r.criterion for r in rs.rubrics}
        for rid, _reason, _crit in rep.failed:
            key = (rid, text_by_id.get(rid, rid))
            counts[key] = counts.get(key, 0) + 1
    ranked = sorted(counts.items(), key=lambda kv: -kv[1])[:limit]
    return [(criterion, n) for (_rid, criterion), n in ranked]

In [ ]:
for name, info in JUDGE_BIASES.items():
    print(f"● {name}")
    print(f"    問題：{info['說明']}")
    print(f"    緩解：{info['緩解']}\n")

#### 步驟一：為每個商品產生驗收清單

這一步和被評的文案無關，只看商品資料。

In [ ]:
rubric_sets = {p['sku']: generate_rubrics(client, p, ledger) for p in PRODUCTS}

demo = rubric_sets[PRODUCTS[0]['sku']]
print(f"{PRODUCTS[0]['name']} 的驗收清單（{len(demo.rubrics)} 條）\n")
for r in demo.rubrics:
    mark = "★" if r.critical else " "
    print(f"  {mark} {r.id}  [{r.dimension}] {r.criterion}")
print("\n★ = critical，未通過就不該上架")

#### 步驟二：逐條檢查

只送規則層過關的文案去評審 —— **這是省錢的關鍵**。
對一份已經違反法規的文案做語意檢查沒有意義，而 judge 用的是比生成更貴的模型。

In [ ]:
rubric_reports = []
skipped = 0
by_key = {(r.sku, r.version): r for r in rule_results}

for product in PRODUCTS:
    for version in PROMPT_VERSIONS:
        rr = by_key[(product['sku'], version)]
        if not should_judge(rr):
            skipped += 1
            continue
        rubric_reports.append(check_against_rubrics(
            client, outputs[(product['sku'], version)],
            rubric_sets[product['sku']], version, ledger
        ))

print(f"送審 {len(rubric_reports)} 筆，規則層已擋下 {skipped} 筆（省下的就是錢）\n")
pd.DataFrame(summarize_rubrics(rubric_reports)).T

#### 最常沒過的判準 —— 這才是能拿去改 prompt 的東西

這正是二元 rubric 勝過分數的地方：「3.8 分」不能行動，
「有 9 個商品沒寫出含量」可以。

In [ ]:
for criterion, n in most_common_failures(rubric_reports, rubric_sets):
    print(f"  {n:>2} 次未通過 — {criterion}")

worst = [r for r in rubric_reports if r.version == 'v0' and r.failed]
if worst:
    r = worst[0]
    print(f"\n{r.sku} v0 未通過的細節（{r.pass_rate}% 通過）：")
    for rid, reason, crit in r.failed[:4]:
        print(f"  {'★' if crit else ' '} {rid}: {reason}")

### ★★ 合併對比表 —— 每一版修好一件事

這張表是整份 notebook 的結論。左邊三欄由 v1 修好、中間兩欄由 v2 修好、
最後一欄由 v3 修好 —— **每一版的貢獻都能單獨看見**。

In [ ]:
print(significance_note(len(PRODUCTS), 7))

combined = combined_table(rule_results, summarize_rubrics(rubric_reports))
combined.style.format("{:.1f}%", na_rep="—").background_gradient(
    cmap="RdYlGn", vmin=0, vmax=100)

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **38:00 — ★ 全場高潮。表格出來後停 5 秒不要說話，讓大家看。**
>
> 然後：「每一版修好一件事。左邊三欄是 v1、中間兩欄是 v2、最後一欄是 v3。」
>
> **接著一定要講這兩句，這是誠信也是專業度：**
>
> 1.「v3 在 rubric 上沒有贏過 v2 —— 差距在誤差內。但可機器讀取那一欄
>    從 0 變成 100%，那才是分水嶺。」
> 2.「而且這只有 12 個商品。兩三個百分點的差距是雜訊，不是改善。
>    我不會拿這種差距去說服任何人。」
>
> **不要**說「你看全部都變好了」。台下有工程師，會扣分。

</div>

#### 讀這張表的時候要小心兩件事

**一、樣本很小。** 12 個商品，rubric 每個 7 條，總共 84 次檢查。
翻轉一次檢查就是 1.2 個百分點。所以**兩三個百分點的差距是雜訊，不是改善**。
把雜訊當成訊號，是評測工作最容易犯、也最傷的錯。

**二、v3 在 rubric 上不一定贏過 v2。** 結構化輸出會犧牲一點文字彈性，
兩者常常在誤差內打平。但 v3 贏在**可機器讀取那一欄從 0% 變成 100%** ——
這是真正的分水嶺，因為前面幾版根本進不了系統。

這是真實的工程取捨，不是缺陷。一張每格都完美遞增的表，通常代表有人在調數字。

## §4 場景 B：評論洞察

方法論完全相同，方向相反：

    場景 A   結構化資料 → 非結構化文字   （生成）
    場景 B   非結構化文字 → 結構化資料   （抽取）

**這一節的目的不是教評論分析**，而是證明你剛學的是一套流程，不是一個文案技巧。
同樣要 schema、同樣要評測、同樣要算成本。

In [ ]:
"""場景 B：評論洞察分析。

方法論與場景 A 完全相同，方向相反：
    場景 A  結構化資料 → 非結構化文字   （生成）
    場景 B  非結構化文字 → 結構化資料   （抽取）

這一節的重點不是評論分析本身，而是證明**這是一套可遷移的流程，
不是一個文案技巧**。同樣要 schema、同樣要評測、同樣要算成本。

商業價值一句話：把「客服每天讀 500 則評論」變成「BigQuery 一句 SQL」。
"""

from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from datetime import UTC


ASPECTS = ["產品品質", "價格", "物流配送", "包裝", "客服", "使用體驗", "其他"]

REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "sentiment": {
            "type": "string",
            "enum": ["positive", "neutral", "negative"],
        },
        "aspects": {
            "type": "array",
            "items": {"type": "string", "enum": ASPECTS},
            "description": "這則評論實際談到的面向，可多選",
        },
        "is_about_product": {
            "type": "boolean",
            "description": "評論主體是商品本身，還是物流／客服等非商品因素",
        },
        "actionable_suggestion": {
            "type": "string",
            "description": "可執行的改善建議；若無則填空字串",
        },
        "urgency": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": [
        "sentiment",
        "aspects",
        "is_about_product",
        "actionable_suggestion",
        "urgency",
    ],
}

EXTRACT_PROMPT = """你是電商營運分析師，要把客戶評論轉成結構化資料進資料倉儲。

【重要判準】
- `is_about_product`：評論在抱怨物流慢、客服態度、包裝破損時，**主體不是商品**，
  應為 false。這一欄的用途是把「商品不好」和「服務不好」分開 ——
  兩者要交給不同部門處理，混在一起會導致採購端誤判商品品質。
- `urgency`：只有涉及安全疑慮、法規風險、或大量客戶可能受影響時才是 high。
  單純的個人喜好不合不算。
- `actionable_suggestion`：必須是團隊真的能執行的動作（改包裝、補說明、調規格），
  不要寫「提升品質」這種無法執行的空話。無明確建議就留空字串。

【評論】
商品：{product_name}
評分：{rating} 星
內容：{text}

請依 schema 輸出。"""


@dataclass
class ReviewInsight:
    review_id: str
    sku: str
    rating: int
    sentiment: str
    aspects: list[str]
    is_about_product: bool
    actionable_suggestion: str
    urgency: str
    error: str | None = None


def extract_one(client, review: dict, product_name: str, ledger=None) -> ReviewInsight:
    """把單則評論轉成結構化資料。"""
    import time

    from google.genai import types


    prompt = EXTRACT_PROMPT.format(
        product_name=product_name,
        rating=review["rating"],
        text=review["text"],
    )

    started = time.time()
    try:
        resp = client.models.generate_content(
            model=GEN_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,  # 抽取任務要可重現，不要創意
                response_mime_type="application/json",
                response_schema=REVIEW_SCHEMA,
            ),
        )
        meta = resp.usage_metadata
        if ledger is not None:
            ledger.record(
                "gen:insight",
                Usage(
                    model=GEN_MODEL,
                    input_tokens=getattr(meta, "prompt_token_count", 0) or 0,
                    output_tokens=getattr(meta, "candidates_token_count", 0) or 0,
                    latency_s=round(time.time() - started, 2),
                ),
            )
        data = json.loads(resp.text)
        return ReviewInsight(
            review_id=review["review_id"],
            sku=review["sku"],
            rating=review["rating"],
            sentiment=data["sentiment"],
            aspects=list(data["aspects"]),
            is_about_product=bool(data["is_about_product"]),
            actionable_suggestion=data["actionable_suggestion"],
            urgency=data["urgency"],
        )
    except Exception as e:  # noqa: BLE001
        return ReviewInsight(
            review_id=review["review_id"],
            sku=review["sku"],
            rating=review["rating"],
            sentiment="neutral",
            aspects=[],
            is_about_product=True,
            actionable_suggestion="",
            urgency="low",
            error=str(e),
        )


def validate_insights(insights: list[ReviewInsight]) -> dict:
    """場景 B 也要評測 —— 這是本節的重點。

    抽取任務沒有「好文案」這種主觀標準，但仍有可自動檢查的規則：
    - enum 值是否合法
    - 星等與情緒是否矛盾（1 星卻標 positive，通常是抽取錯誤）
    - 高星等卻標 high urgency（同樣可疑）

    這些規則抓到的不是「模型寫得不好」，而是「模型理解錯了」—— 更嚴重。
    """
    issues = []
    for r in insights:
        if r.error:
            issues.append((r.review_id, f"抽取失敗：{r.error}"))
            continue
        if r.sentiment not in {"positive", "neutral", "negative"}:
            issues.append((r.review_id, f"sentiment 值非法：{r.sentiment}"))
        if bad := [a for a in r.aspects if a not in ASPECTS]:
            issues.append((r.review_id, f"aspects 出現未定義值：{bad}"))
        if r.rating <= 2 and r.sentiment == "positive":
            issues.append((r.review_id, f"{r.rating} 星卻判為 positive，疑似抽取錯誤"))
        if r.rating >= 4 and r.sentiment == "negative":
            issues.append((r.review_id, f"{r.rating} 星卻判為 negative，疑似抽取錯誤"))
        if r.rating == 5 and r.urgency == "high":
            issues.append((r.review_id, "5 星卻標記 high urgency，請人工確認"))

    return {
        "總筆數": len(insights),
        "問題筆數": len(issues),
        "通過率": round(100 * (len(insights) - len(issues)) / max(len(insights), 1), 1),
        "明細": issues,
    }


def to_dataframe(insights: list[ReviewInsight]):
    """轉成 DataFrame。欄位刻意設計成可直接進 BigQuery 的扁平結構。"""
    import pandas as pd

    rows = []
    for r in insights:
        d = asdict(r)
        d["aspects"] = "|".join(r.aspects)  # BigQuery 可用 SPLIT() 還原
        rows.append(d)
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# BigQuery —— 把「AI 當 ETL」這件事真的做完
# --------------------------------------------------------------------------
BQ_SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS `{project}.{dataset}.{table}` (
  review_id             STRING  NOT NULL,
  sku                   STRING  NOT NULL,
  rating                INT64,
  sentiment             STRING,          -- positive / neutral / negative
  aspects               ARRAY<STRING>,   -- 一則評論可能談到多個面向
  is_about_product      BOOL,            -- 把商品問題與服務問題分開的關鍵欄位
  actionable_suggestion STRING,
  urgency               STRING,          -- low / medium / high
  ingested_at           TIMESTAMP
)
PARTITION BY DATE(ingested_at)
CLUSTER BY sku, sentiment
"""

# 這句 SQL 就是整個場景 B 的商業價值：客服原本要讀 500 則評論，現在是一句查詢。
BQ_INSIGHT_SQL = """
SELECT
  sku,
  COUNTIF(is_about_product AND sentiment = 'negative') AS 商品負評,
  COUNTIF(NOT is_about_product AND sentiment = 'negative') AS 服務負評,
  COUNTIF(urgency = 'high')                             AS 高急迫,
  ROUND(AVG(rating), 2)                                 AS 平均星等
FROM `{project}.{dataset}.{table}`
GROUP BY sku
HAVING 商品負評 > 0 OR 服務負評 > 0
ORDER BY 商品負評 DESC, 服務負評 DESC
"""


def load_to_bigquery(insights: list[ReviewInsight], verbose: bool = True):
    """把抽取結果寫進 BigQuery。

    這一步讓「非結構化文字 → 結構化資料」這句話變成真的 —— 資料真的落地、
    真的能被 SQL 查詢、真的能接上既有的 BI 報表。

    schema 的設計重點在 `is_about_product`：把「商品不好」與「服務不好」
    分開儲存。混在一起統計，採購會以為商品品質有問題，實際上是物流慢。
    """
    from datetime import datetime

    from google.cloud import bigquery

    client = bigquery.Client(project=PROJECT_ID)
    ds_id = f"{PROJECT_ID}.{BQ_DATASET}"

    ds = bigquery.Dataset(ds_id)
    ds.location = BQ_LOCATION
    client.create_dataset(ds, exists_ok=True)

    table_ref = f"{ds_id}.{BQ_TABLE}"
    client.query(
        BQ_SCHEMA_SQL.format(
            project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE
        )
    ).result()

    now = datetime.now(UTC).isoformat()
    rows = [
        {
            "review_id": r.review_id,
            "sku": r.sku,
            "rating": r.rating,
            "sentiment": r.sentiment,
            "aspects": r.aspects,
            "is_about_product": r.is_about_product,
            "actionable_suggestion": r.actionable_suggestion,
            "urgency": r.urgency,
            "ingested_at": now,
        }
        for r in insights
        if not r.error
    ]
    if not rows:
        raise RuntimeError("沒有可寫入的資料（全部抽取失敗）")

    errors = client.insert_rows_json(table_ref, rows)
    if errors:
        raise RuntimeError(f"BigQuery 寫入失敗：{errors}")

    if verbose:
        print(f"✓ 已寫入 {len(rows)} 列 → {table_ref}")
        print("  分區：DATE(ingested_at)　叢集：sku, sentiment")
    return table_ref


def query_bigquery_insights():
    """跑那句「取代人工讀 500 則評論」的 SQL。"""
    from google.cloud import bigquery

    client = bigquery.Client(project=PROJECT_ID)
    sql = BQ_INSIGHT_SQL.format(
        project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE
    )
    return client.query(sql).to_dataframe()


def business_summary(insights: list[ReviewInsight]) -> str:
    """重點在這裡：資料變成決策。"""
    ok = [r for r in insights if not r.error]
    if not ok:
        return "無有效資料"

    product_issues = [r for r in ok if r.is_about_product and r.sentiment == "negative"]
    service_issues = [
        r for r in ok if not r.is_about_product and r.sentiment == "negative"
    ]
    urgent = [r for r in ok if r.urgency == "high"]

    aspect_counts: dict[str, int] = {}
    for r in ok:
        if r.sentiment == "negative":
            for a in r.aspects:
                aspect_counts[a] = aspect_counts.get(a, 0) + 1

    top = sorted(aspect_counts.items(), key=lambda kv: -kv[1])[:3]
    suggestions = [r.actionable_suggestion for r in ok if r.actionable_suggestion]

    lines = [
        f"有效評論 {len(ok)} 則",
        "",
        f"  商品相關負評   {len(product_issues):>3} 則  → 採購／研發要看",
        f"  服務相關負評   {len(service_issues):>3} 則  → 物流／客服要看",
        f"  高急迫案件     {len(urgent):>3} 則  → 今天就要處理",
        "",
        "負評集中的面向：",
    ]
    lines += [f"  {a}：{c} 則" for a, c in top] or ["  （無）"]
    lines += ["", f"可執行建議 {len(suggestions)} 條，例如："]
    lines += [f"  - {s}" for s in suggestions[:3]]
    lines += [
        "",
        "↑ 這就是價值所在：把「客服每天讀 500 則評論」變成一句 SQL。",
        "  注意『商品負評』與『服務負評』已經分開 —— 混在一起會讓採購誤判商品品質。",
    ]
    return "\n".join(lines)

In [ ]:
REVIEWS = {
 "_meta": {
  "description": "示範用假評論資料，模擬台灣電商評論區的真實樣態：長短不一、有錯字、有離題、有物流抱怨混在商品評價裡。",
  "note": "刻意保留雜訊（注音、錯字、只有標點的短評、抱怨物流而非商品），用來展示模型在髒資料上的表現，這才是真實情況。",
  "target_schema": {
   "sentiment": "positive | neutral | negative",
   "aspects": "list of 產品品質 | 價格 | 物流配送 | 包裝 | 客服 | 使用體驗 | 其他",
   "is_about_product": "bool — 用來把物流客訴從商品評價中分離",
   "actionable_suggestion": "string | null",
   "urgency": "low | medium | high"
  }
 },
 "reviews": [
  {
   "review_id": "R001",
   "sku": "HB-1001",
   "rating": 5,
   "text": "吃了快兩個月，本來只是想說試試看，結果現在每天都會記得吃。膠囊不大顆很好吞，也沒有什麼味道。會回購。"
  },
  {
   "review_id": "R002",
   "sku": "HB-1001",
   "rating": 2,
   "text": "東西還沒拆就先說，寄來的時候盒子整個壓扁了…雖然裡面膠囊看起來沒事但這樣送禮很尷尬耶，希望包裝可以加強"
  },
  {
   "review_id": "R003",
   "sku": "HB-1002",
   "rating": 4,
   "text": "粉末沖水很好溶解不會結塊這點加分，味道偏淡幾乎沒味道我可以接受。唯一想抱怨的是一盒才30包但價格快1200，有點貴"
  },
  {
   "review_id": "R004",
   "sku": "HB-1002",
   "rating": 1,
   "text": "下單到收到花了九天！！！客服問了三次都說在處理中，最後也沒給我明確時間。產品本身還沒吃不評論，但這個出貨速度真的不行"
  },
  {
   "review_id": "R005",
   "sku": "HB-1003",
   "rating": 5,
   "text": "蔓越莓口味比我想像中好喝很多，不會有腥味這點很重要！！睡前泡一杯變成習慣了"
  },
  {
   "review_id": "R006",
   "sku": "HB-1003",
   "rating": 3,
   "text": "還可以吧"
  },
  {
   "review_id": "R007",
   "sku": "CS-2001",
   "rating": 5,
   "text": "敏感肌用了三週沒有刺痛感，這對我來說已經是很高的評價了。質地是水感的很好推開，上妝前用也不會搓泥。唯一小缺點是滴管有時候會沾到瓶口"
  },
  {
   "review_id": "R008",
   "sku": "CS-2001",
   "rating": 2,
   "text": "我用了長痘痘...不知道是不是我不適合，成分表看起來是很溫和沒錯啦。退貨流程倒是很順利客服人很好"
  },
  {
   "review_id": "R009",
   "sku": "CS-2002",
   "rating": 4,
   "text": "泡沫很綿密洗完不緊繃，但120ml用得有點快，希望出大容量版本"
  },
  {
   "review_id": "R010",
   "sku": "CS-2003",
   "rating": 1,
   "text": "膜布跟我臉根本不合，眼睛下面那塊一直翹起來，敷到一半就掉了。精華液倒是蠻多的但都流到脖子去了= ="
  },
  {
   "review_id": "R011",
   "sku": "EL-3001",
   "rating": 5,
   "text": "降噪真的有感，捷運上開最強模式幾乎聽不到廣播聲。續航也很夠用，通勤來回一週充一次盒子就好。雙裝置切換超方便，手機筆電來回切不用重連"
  },
  {
   "review_id": "R012",
   "sku": "EL-3001",
   "rating": 3,
   "text": "音質沒話說但戴久了耳朵會痛，我耳道比較小可能不適合。另外通話時對方說我這邊有雜音，不知道是不是個案"
  },
  {
   "review_id": "R013",
   "sku": "EL-3002",
   "rating": 5,
   "text": "體積真的小！！之前那顆65W是這個的兩倍大。三個孔同時插筆電手機耳機都還能跑滿速，出差只帶這顆就夠"
  },
  {
   "review_id": "R014",
   "sku": "EL-3003",
   "rating": 2,
   "text": "吸力是不錯啦可是集塵盒也太小了吧0.5L根本掃不到一半就要倒，而且倒的時候灰塵會噴出來設計有問題"
  },
  {
   "review_id": "R015",
   "sku": "FD-4001",
   "rating": 4,
   "text": "耶加雪菲的花香有出來，掛耳能做到這樣不容易。不過我覺得建議沖法的水溫寫得有點模糊，第一次沖太燙有澀感，後來自己調到88度才對"
  }
 ]
}['reviews']

name_by_sku = {p['sku']: p['name'] for p in PRODUCTS}
insights = [extract_one(client, r, name_by_sku.get(r['sku'], '未知商品'), ledger)
            for r in REVIEWS]

to_dataframe(insights)[['review_id','rating','sentiment','aspects',
                        'is_about_product','urgency']]

### 抽取任務也要評測

沒有「好文案」這種主觀標準，但仍有可自動檢查的規則：
星等與情緒矛盾、enum 值非法、5 星卻標高急迫。

這些抓到的不是「模型寫得不好」，而是**「模型理解錯了」—— 更嚴重**。

In [ ]:
check = validate_insights(insights)
print(f"通過率 {check['通過率']}%  （{check['總筆數'] - check['問題筆數']}/{check['總筆數']}）")
for rid, msg in check['明細']:
    print(f"  ⚠ {rid}: {msg}")

print("\n" + "=" * 60)
print(business_summary(insights))

#### 落地到 BigQuery — 讓「變成結構化資料」不只是說說

到這裡為止，結構化資料還只存在於記憶體裡。真正的價值要等它**進到資料倉儲、
能被 SQL 查詢、能接上既有的 BI 報表**才會發生。

注意 schema 的設計重點：`is_about_product` 這個欄位把「商品不好」與
「服務不好」分開儲存。混在一起統計，採購會以為商品品質有問題，
實際上是物流慢。

> 預設 `USE_BIGQUERY = False`。要真的寫入請在 CONFIG 打開，
> 並確認專案已啟用 BigQuery API 且帳號有 `bigquery.dataEditor` 權限。
> 離線重播模式不涵蓋 BigQuery 呼叫。

In [ ]:
print("建表 SQL：")
print(BQ_SCHEMA_SQL.format(project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE))
print("\n分析 SQL —— 這句就是整個場景 B 的商業價值：")
print(BQ_INSIGHT_SQL.format(project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE))

In [ ]:
if USE_BIGQUERY:
    load_to_bigquery(insights)
    bq_df = query_bigquery_insights()
    display(bq_df)
else:
    print("USE_BIGQUERY = False — 只展示 schema 與 SQL，未實際寫入。")
    print("要真的跑，請在 CONFIG 把 USE_BIGQUERY 設為 True。")

## §5 成本

**教公式，不背數字。**

    單次成本 = (input_tokens × 單價_in + output_tokens × 單價_out) / 1,000,000
    月成本  = 單次 × SKU數 × 每SKU重生成次數 × (1 + judge 開銷比)

下面印出來的是**這次執行的真實花費**，比任何通用估算都準 ——
因為 prompt 長度、輸出長度、中文 token 密度全是這個專案的實際值。

In [ ]:
"""成本估算：從 demo 的實際用量外推到正式營運規模。

原則：**用公式，不要背數字。**
模型與價格每季在變（例如 Gemini 2.5 Flash-Lite 已公告 2026-10-16 退役），
任何寫死的價格很快就會過期。公式不會。

公式是穩定的，價格常數集中在 config，實際金額由執行結果給出。
"""

from __future__ import annotations

from dataclasses import dataclass



@dataclass
class ScaleEstimate:
    """把 demo 用量外推到正式規模。"""

    sku_count: int
    regen_per_sku: float          # 每個 SKU 平均重新生成幾次（改版、A/B、季節性換檔）
    judge_ratio: float            # 送 judge 的比例（規則層擋掉的不送）
    gen_cost_per_item: float      # 單次生成成本 USD
    judge_cost_per_item: float    # 單次評審成本 USD

    @property
    def total_generations(self) -> float:
        return self.sku_count * self.regen_per_sku

    @property
    def gen_cost(self) -> float:
        return self.total_generations * self.gen_cost_per_item

    @property
    def judge_cost(self) -> float:
        return self.total_generations * self.judge_ratio * self.judge_cost_per_item

    @property
    def total_usd(self) -> float:
        return self.gen_cost + self.judge_cost

    @property
    def total_twd(self) -> float:
        return self.total_usd * USD_TO_TWD

    def render(self) -> str:
        return (
            f"規模假設：{self.sku_count:,} 個 SKU × 每個重生成 {self.regen_per_sku} 次"
            f" = {self.total_generations:,.0f} 次生成\n"
            f"           其中 {self.judge_ratio:.0%} 通過規則層、需要送 LLM 評審\n\n"
            f"  生成成本   {self.total_generations:>10,.0f} 次 × US${self.gen_cost_per_item:.6f}"
            f"  = US${self.gen_cost:>10,.2f}\n"
            f"  評審成本   {self.total_generations * self.judge_ratio:>10,.0f} 次 "
            f"× US${self.judge_cost_per_item:.6f}  = US${self.judge_cost:>10,.2f}\n"
            f"  {'─' * 58}\n"
            f"  合計                                       US${self.total_usd:>10,.2f}"
            f"  （約 NT${self.total_twd:,.0f}）"
        )


def estimate_from_ledger(
    ledger,
    sku_count: int = 100_000,
    regen_per_sku: float = 1.5,
    judge_ratio: float = 0.7,
) -> ScaleEstimate:
    """用 demo 實際跑出來的平均單次成本外推。

    這比任何通用估算都準，因為 prompt 長度、輸出長度、中文 token 密度
    全部是這個專案的真實值，不是別人部落格上的數字。
    """
    gen_calls = [u for t, u in ledger.calls if t.startswith("gen")]
    judge_calls = [u for t, u in ledger.calls if t.startswith("judge")]

    gen_avg = sum(u.cost_usd for u in gen_calls) / len(gen_calls) if gen_calls else 0.0
    judge_avg = (
        sum(u.cost_usd for u in judge_calls) / len(judge_calls) if judge_calls else 0.0
    )

    return ScaleEstimate(
        sku_count=sku_count,
        regen_per_sku=regen_per_sku,
        judge_ratio=judge_ratio,
        gen_cost_per_item=gen_avg,
        judge_cost_per_item=judge_avg,
    )


# --------------------------------------------------------------------------
# 降本四招
# --------------------------------------------------------------------------
LEVERS = [
    {
        "名稱": "模型選型降級",
        "作法": "簡單任務改用更小的模型；只有需要判斷力的環節才用大模型。",
        "省下": "視價差而定，通常是最大的一筆",
        "代價": "品質下降風險 —— **必須有評測表才敢降級**，否則是在賭。",
        "適用": "文案生成這種格式固定、判準明確的任務，通常降級後評測分數不會掉。",
    },
    {
        "名稱": "Batch API",
        "作法": "非即時的工作改走批次介面。商品文案本來就不需要即時。",
        "省下": "約 5 折",
        "代價": "延遲從秒級變成小時級。",
        "適用": "初次匯入全站商品、季節性全面改版 —— 這些場景根本不在乎即時性。",
    },
    {
        "名稱": "Context caching",
        "作法": "把共用的前綴（品牌指南、法規禁詞清單、few-shot 範例）快取起來。",
        "省下": "快取命中部分的 input token 大幅折扣",
        "代價": "有最低 token 門檻與存活時間，前綴要夠長、呼叫要夠密集才划算。",
        "適用": "本專案的 v2/v3 prompt 有大量固定前綴，是典型適用情境。",
    },
    {
        "名稱": "結果快取",
        "作法": "商品規格沒變就不要重新生成。用規格的 hash 當 key。",
        "省下": "取決於商品異動率，通常這是最被低估的一招",
        "代價": "幾乎沒有，只需要一個 key-value 存放。",
        "適用": "所有場景。很多團隊每天全量重跑，其實九成商品根本沒變。",
    },
]


def render_levers() -> str:
    out = []
    for i, lever in enumerate(LEVERS, 1):
        out.append(
            f"{i}. {lever['名稱']}（省下：{lever['省下']}）\n"
            f"   作法：{lever['作法']}\n"
            f"   代價：{lever['代價']}\n"
            f"   適用：{lever['適用']}"
        )
    return "\n\n".join(out)


def measure_chinese_token_density(client, samples: list[str]) -> dict:
    """實測中文的 token 密度，不要用經驗法則。

    直接量出來。中文的 token 密度會隨內容（純中文／中英混雜／
    含數字規格）明顯變動，背一個「一字約 X token」的數字是不可靠的。
    """

    rows = []
    for s in samples:
        tokens = count_tokens(client, s)
        chars = len(s)
        rows.append(
            {
                "字元數": chars,
                "token數": tokens,
                "每字token": round(tokens / chars, 3) if chars else 0,
                "預覽": s[:24] + ("..." if len(s) > 24 else ""),
            }
        )
    densities = [r["每字token"] for r in rows if r["每字token"]]
    return {
        "明細": rows,
        "平均每字token": round(sum(densities) / len(densities), 3) if densities else 0,
        "最低": min(densities) if densities else 0,
        "最高": max(densities) if densities else 0,
    }


def price_check_reminder() -> str:
    """執行前要印出來的提醒。"""
    return (
        f"⚠ 價格常數最後查證時間：{PRICE_LAST_CHECKED}\n"
        f"  上場前請到 https://cloud.google.com/vertex-ai/generative-ai/pricing "
        f"確認並更新 PRICING，\n"
        f"  然後把 PRICE_LAST_CHECKED 改成當天日期。\n"
        f"  不要引用固定的價格數字，以本 notebook 實際印出的金額為準。"
    )

In [ ]:
print(price_check_reminder())
print("\n" + "=" * 60)
print("這次 demo 的實際用量")
print("=" * 60)
print(ledger.summary())

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **41:00 — 唸出實際的「評審佔總成本 __%」。**
>
> 這個數字每次跑都會變，**唸畫面上的，不要唸背下來的**。
> （前一天錄製時先看過一次，心裡有個底。）
>
> 「評審佔了這麼多。這是最常被漏算的一筆 —— 大家算成本只算生成。」

</div>

### 實測中文 token 密度 — 不要背經驗法則

In [ ]:
samples = [
    PRODUCTS[0]['name'],
    "游離型葉黃素30mg，吸收更直接",
    PROMPT_VERSIONS['v3'](PRODUCTS[0],
                          banned_terms=get_banned_terms_for(PRODUCTS[0], BANNED_DATA),
                          tone_examples=TONE_EXAMPLES[PRODUCTS[0]['brand']])[:500],
    "Volta 240W USB-C 編織線 2M 支援40Gbps傳輸",   # 中英數混雜
]
density = measure_chinese_token_density(client, samples)
pd.DataFrame(density['明細'])

In [ ]:
print(f"平均每字 {density['平均每字token']} tokens"
      f"（範圍 {density['最低']} – {density['最高']}）")
print("\n↑ 注意範圍有多寬。中英數混雜的文字密度和純中文差很多，")
print("  所以『中文一個字約 X 個 token』這種經驗法則不可靠，要自己量。")

### 外推到正式規模

In [ ]:
est = estimate_from_ledger(ledger, sku_count=100_000, regen_per_sku=1.5, judge_ratio=0.7)
print(est.render())

### 降本四招

In [ ]:
print(render_levers())

---

## §6 錄製 fixtures（需要離線重播時才執行）

這一格把剛剛所有真實 API 輸出存成檔案，之後就能零網路重播。

**流程：**

1. 在**有網路**的環境把 `RECORD_FIXTURES` 設為 `True`，Run all
2. 執行下面這格，下載 `demo_outputs.json`
3. 把檔案放到 `poc/data/`，執行 `python3 poc/build_notebook.py` 重新產生 notebook
4. 把 `OFFLINE_MODE` 設為 `True`，再 Run all 驗證一次 —— **這次應該完全不連網**
5. 之後在任何沒有網路的環境都能完整重跑

In [ ]:
if RECORD_FIXTURES:
    dump = client.dump()
    with open(FIXTURES_FILE, "w", encoding="utf-8") as f:
        json.dump(dump, f, ensure_ascii=False)
    print(f"✓ 已錄製 {len(dump['calls']):,} 筆呼叫、"
          f"{len(dump['token_counts']):,} 筆 token 計數 → {FIXTURES_FILE}")
    try:
        from google.colab import files
        files.download(FIXTURES_FILE)
    except ImportError:
        print("  （本機執行，檔案已寫在當前目錄）")
elif OFFLINE_MODE:
    print(f"離線重播模式：本次共命中 {client.models.hits:,} 筆錄製輸出，全程未連網。")
else:
    print("目前是一般連線模式。若要錄製以供離線重播，請把 RECORD_FIXTURES 設為 True。")

---

## 收尾

回到一開始的問題：**你怎麼知道它有沒有變好？**

現在你有一張表可以回答。而且那張表是自動產生的，
下次改 prompt、換模型、降級省錢的時候，重跑一次就知道有沒有退步。

**這才是能上線的東西。**

---

### 帶回去的三件事

1. **先建規則層。** 免費、確定性、可以進 CI。不要一開始就做 LLM judge。
2. **用 API 原生的 structured output**，不要用 prompt 硬凹 JSON。
3. **評測本身要花錢**，記得算進去 —— 這是最常被漏掉的一筆。

### 90 天導入路徑

| 階段 | 時間 | 產出 |
|---|---|---|
| Vertex AI Studio 驗證可行性 | 2 週 | 這件事到底做不做得成 |
| 建 golden set + API 串接 | 4 週 | 30–50 筆就足以開始 |
| 小流量上線 + 監控 | 6 週 | 有數字可以對董事會報告 |

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **41:45 — 捲回對比表收尾。**
>
> 「所以：你怎麼知道它有沒有變好？看這張表。」
>
> 然後切回投影片，講最後三件事帶回去。
>
> ---
>
> ⏱ **超時的話砍 §4（場景 B）**，不影響主論點。

</div>